# Two-Tower Model — complementary products

**The goal of this notebook is to build a two-tower neural network (TTN) that
finds complementary products** — given an item a user is looking at, retrieve
the items that are bought *alongside* it rather than the items most similar to
it. A phone case complements a phone; another phone does not.

The approach follows **[Suggest, complement, inspire: story of Two Tower
recommendations at Allegro.com](https://arxiv.org/html/2508.03702v1)**
(Osowska-Kurczab, Nazarko, Marzec, Wojciechowska & Kremeňová, RecSys '25),
whose Complementary-TT model is the architecture this work is based on.

## Both towers describe items

This is the part that differs from the classic user/item two-tower setup, and
it shapes every column decision below: **the query tower and the candidate
tower both consume item information.** Neither tower is a user tower.

```
   query ITEM features                    candidate ITEM features
        │                                          │
   ┌────▼────┐                                ┌────▼────┐
   │  QUERY  │  product encoder               │CANDIDATE│  product encoder
   │  TOWER  │  (+ target category)           │  TOWER  │
   └────┬────┘                                └────┬────┘
        │                                          │
   q ∈ ℝ^d  ──────────  score = q · c  ──────────  c ∈ ℝ^d
```

Both towers share the same *architecture* — the paper's "Product Encoder":
each item attribute goes through its own embedding table, the vectors are
concatenated, passed through an MLP and L2-normalised. In the paper the query
tower is the only one modified for the complementary task: the query product
embedding is concatenated with a **target category embedding** drawn from a
one-to-many complementary-category mapping, while the candidate tower stays a
plain product encoder.

That mapping is what `complementary_cats_pairs/` produces —
`data/complementary_categories.pkl`, source category path → target category path,
scored by support and lift. §1 loads it. The co-purchase pairs that supply the
training positives come from the same package's `pairs.ipynb`.

Because both sides are items, per-user history is not a tower input at all and
this notebook does not load it. A user's history still shapes the data — it is
what defines which items count as co-purchased — but that work happens upstream,
in `complementary_cats_pairs/pairs.ipynb`.

## What this notebook covers

It loads the tables the model needs and turns them into training pairs. §1
reads everything; §4 builds the pair tables and §8 finishes them as
`tower_pairs_train` and `tower_pairs_test`, one row per directed
(query item, candidate item) example:

| Column | |
| --- | --- |
| `asin_query`, `query_cat_2/3/4` | the item being looked at |
| `asin_target`, `target_cat_2/3/4` | an item bought alongside it, whose category the mapping licenses |

The train/test split is temporal and was applied when `pairs.ipynb` built the
two co-purchase tables, on `date_threshold` from `ttn/constants.json`. This
notebook reads both and carries them through the same steps, producing
`tower_pairs_train` and `tower_pairs_test`.

Every statistic is **fitted on the training slice and applied to both** — the
brand fold, the eight categorical vocabularies, the price medians and the
`target_node` vocabulary. Fitting any of them twice would leave the same
category on a different integer code at evaluation time, and that code is the
embedding index. Model definition and training come next.


In [78]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# data/ lives at the repo root, one level up from this ttn/ folder
ROOT = Path("..").resolve()
DATA_DIR = ROOT / "data"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Show every column/variable when displaying a dataframe
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 50)
# Turn off scientific notation (e.g. 2.447268e+06 -> 2447268.00)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Load the datasets

Everything this notebook reads, in one place. Nothing below this section opens
a file — every later cell transforms tables that are already in memory.

| Dataset | Grain | What it is |
| --- | --- | --- |
| `Home_and_Kitchen_filtered.csv` | one row per review | the interaction log — who bought what, when |
| `df_features.pkl` | one row per `asin` | extracted item attributes: `cat_*`, `brand`, `title_cleaned`, the per-field columns and their parsed measures |
| `co_purchase_pairs_train.pkl` | one row per item pair | pairs from interactions **before** `date_threshold` — the training positives |
| `co_purchase_pairs_test.pkl` | one row per item pair | pairs from interactions **at or after** it — the held-out positives |
| `complementary_categories.pkl` | one row per directed category pair | which category buys into which, filtered by support and lift — the **complementary mapping** |
| `meta_Home_and_Kitchen_filtered.csv` | one row per `asin` | the *unfiltered* catalogue; describes 150,826 items `df_features` does not |

`asin` and `reviewerID` are pinned to `str` throughout so ids with leading
zeros (e.g. `0560467893`) survive the read.

Only five of the catalogue's fifteen columns are read. `df_features` already
carries every field the catalogue has — the catalogue's value here is
**coverage**, not extra columns: it describes 28,537 reviewed asins that have
no `df_features` row, every one of them in a `cat_3` with no extraction schema.
Reading all fifteen columns of a 2.1 GB file to use four of them is waste.

All of it together peaks at about **4.5 GB** of RAM.

Both pair tables are read here and every step below runs over both. The split
is temporal and was applied when `pairs.ipynb` built them, so nothing here
re-derives it — `date_threshold` is read only to scope the statistics §5 and
§7 fit to the training period.


In [79]:
import json
from pathlib import Path

# --- 1. Interactions: one row per review ----------------------------------
df_reviews = pd.read_csv(
    DATA_DIR / "Home_and_Kitchen_filtered.csv",
    dtype={"asin": str, "reviewerID": str},
    low_memory=False,
)

# --- 2. Item features: one row per asin, the extracted attributes ---------
df_features = pd.read_pickle(DATA_DIR / "df_features.pkl")

# --- 3. Co-purchase pairs, one table per side of the cutoff ---------------
co_pairs = {
    "train": pd.read_pickle(DATA_DIR / "co_purchase_pairs_train.pkl"),
    "test": pd.read_pickle(DATA_DIR / "co_purchase_pairs_test.pkl"),
}

# The split point, read from the same file pairs.ipynb reads. Only needed to
# scope the fitted statistics in §5 and §7 to the training period.
DATE_THRESHOLD = json.loads((Path("constants.json")).read_text())["date_threshold"]
cutoff_time = pd.Timestamp(DATE_THRESHOLD).timestamp()

# --- 4. The complementary category mapping --------------------------------
comp_cat = pd.read_pickle(DATA_DIR / "complementary_categories.pkl")

# --- 5. The unfiltered catalogue: coverage for items df_features lacks ----
meta_catalogue = pd.read_csv(
    DATA_DIR / "meta_Home_and_Kitchen_filtered.csv",
    usecols=["asin", "category", "title", "brand", "price"],
    dtype={"asin": str},
    low_memory=False,
)

for name, frame in [
    ("df_reviews", df_reviews), ("df_features", df_features),
    ("co_pairs[train]", co_pairs["train"]), ("co_pairs[test]", co_pairs["test"]),
    ("comp_cat", comp_cat),
    ("meta_catalogue", meta_catalogue),
]:
    print(f"{name:<18} {str(frame.shape):>18}")

print(f"\nunique users: {df_reviews['reviewerID'].nunique():,} | "
      f"unique items: {df_reviews['asin'].nunique():,}")
print(f"items described by df_features : {df_features['asin'].nunique():,}")
print(f"items described by the catalogue: {meta_catalogue['asin'].nunique():,}")
print(f"split point (ttn/constants.json): {DATE_THRESHOLD}")
df_reviews.head(5)

df_reviews              (6898955, 11)
df_features             (1134566, 92)
co_pairs[train]         (10595885, 2)
co_pairs[test]            (942795, 2)
comp_cat                   (5845, 11)
meta_catalogue           (1300540, 5)

unique users: 777,242 | unique items: 189,172
items described by df_features : 1,134,566
items described by the catalogue: 1,285,392
split point (ttn/constants.json): 2017-12-09


,overall,verified,reviewTime,reviewerID,asin,reviewerName,summary,unixReviewTime,vote,style,image
0,5.00,True,"11 5, 2015",A8LUWTIPU9CZB,0560467893,Linda Fahner,Five Stars,1446681600,NaN,NaN,NaN
1,3.00,True,"05 7, 2015",A3B6GKQQ1JJ167,0560467893,Harry Slaughter,Meh,1430956800,2,NaN,NaN
2,5.00,True,"01 22, 2014",A3MCTN65BU7XRA,0681795107,luckyg,Recommend,1390348800,NaN,{'Color:': ' Brushed Stainless'},NaN
3,1.00,True,"10 30, 2013",A7JVZFSXVY9RL,0681795107,Nickleen,Not keeping coffee hot for long enough,1383091200,NaN,{'Color:': ' Brushed Stainless'},NaN
4,1.00,True,"09 20, 2013",A2RQ7VLAK1SHPU,0681795107,Lacemaker427,Leaks like a waterfall when at an angle!,1379635200,NaN,{'Color:': ' Red'},NaN


## 2. Validate the feature table

Before anything is joined, check that `df_features.pkl` is what the pipeline
promised: the exact expected column set, one row per `asin`, numeric columns
actually numeric, coverage above its floors, unit columns free of new values,
and every `_cleaned` column inside its bound.

A **FAIL on `columns`** is the one to care about most — it means a feature
appeared that nothing describes, or one silently disappeared.

In [80]:
from feature_extraction_workflow.validations import run_all

report = run_all(df_features, DATA_DIR / "master_metadata.json")
display(report if len(report) else "no findings")

1,134,566 rows x 92 columns — 1 failure(s), 0 warning(s)


,check,level,subject,detail
0,columns,FAIL,also_buy,present in the table but not in the contract


## 3. Clean the item category path

`cat_4_clean` is built from `data/category_taxonomy.json` — a reviewed
whitelist of which `(cat_3, cat_4)` pairs are real categories rather than
product bullets that leaked into the path. 921 → 451 distinct values, with
0.1% of items landing in a `<cat_3>_Other` bucket.

This runs before anything else because the complementary mapping's `cat_4`
values are folded the same way. Joining §4 on the raw `cat_4` would match on
spelling rather than meaning and drop roughly one valid pair in six.


In [81]:
import json

TAXONOMY_PATH = DATA_DIR / "category_taxonomy.json"
with open(TAXONOMY_PATH) as f:
    taxonomy = json.load(f)

# cat_3 -> the set of cat_4 values that survived the review
valid_cat_4 = {c3: set(vals) for c2 in taxonomy for c3, vals in taxonomy[c2].items()}
print(f"taxonomy: {len(taxonomy)} cat_2 | {len(valid_cat_4)} cat_3 | "
      f"{sum(len(v) for v in valid_cat_4.values())} valid cat_4 slots")

MISSING = "Missing"
OTHER_SUFFIX = "_Other"

cat_3 = df_features["cat_3"].astype(str)
cat_4 = df_features["cat_4"].fillna(MISSING).astype(str)

# A value is kept only if it is valid *under its own parent* — the same label
# can be real in one branch and junk in another.
valid_pairs = {(c3, v) for c3, vals in valid_cat_4.items() for v in vals}
keep = pd.Series(list(zip(cat_3, cat_4)), index=df_features.index).isin(valid_pairs)

df_features["cat_4_clean"] = np.where(keep, cat_4, cat_3 + OTHER_SUFFIX)

n_before = df_features["cat_4"].nunique(dropna=False)
n_after = df_features["cat_4_clean"].nunique()
n_folded = int((~keep).sum())
print(f"\ndistinct cat_4 : {n_before:,} -> {n_after:,}")
print(f"items folded into '<cat_3>{OTHER_SUFFIX}': {n_folded:,} "
      f"({n_folded / len(df_features) * 100:.2f}% of the catalog)")
df_features[["asin", "cat_2", "cat_3", "cat_4", "cat_4_clean"]].head(5)

taxonomy: 7 cat_2 | 69 cat_3 | 521 valid cat_4 slots

distinct cat_4 : 921 -> 451
items folded into '<cat_3>_Other': 1,099 (0.10% of the catalog)


,asin,cat_2,cat_3,cat_4,cat_4_clean
0,0001487795,Kitchen & Dining,Dining & Entertaining,Dinnerware,Dinnerware
1,0002020300,Home Dcor,Candles & Holders,Candles,Candles
2,0006564224,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware,Glassware & Drinkware
3,0009046461,Bath,Bathroom Accessories,None,Missing
4,0234937912,Home Dcor,Home Fragrance,Incense & Incense Holders,Incense & Incense Holders


## 4. Build the pair tables

`build_tower_pairs` turns one co-purchase table into directed (query, target)
rows, and runs over both slices. Three steps:

1. **Join the categories onto both ends.** Inner join on `asin`, so a pair
   survives only if both items have features.
2. **Inner join onto the mapping, in order** — `asinA`'s three categories
   against the mapping's first three (`src_*`), `asinB`'s against the last three
   (`dst_*`). Here `asinA` is the source, so it becomes the **query**.
3. **Inner join again, reversed**, which makes `asinB` the query.

Then concatenate, both halves relabelled so the source side is always
`asin_query` / `query_cat_*`. A pair licensed in both directions appears in both
tables — `X → Y` and `Y → X` are two different examples, not duplicates.

**Nothing here is fitted.** Every step is a per-row join against a fixed
mapping, so running it on the test slice introduces no leakage. Everything that
*is* fitted — the brand threshold, the vocabularies, the price medians —
comes in §§5–8 and is learned from the training slice alone.

Categories stay plain strings at this stage. Making them `category` dtype is
itself a fitting step: the vocabulary decides which integer each value gets, and
that has to be learned once from train and applied to test, never derived twice.


In [82]:
from complementary_cats_pairs import DST_COLS, SRC_COLS

CAT_LEVELS = ["cat_2", "cat_3", "cat_4_clean"]     # matches the mapping's levels
QUERY_CATS = ["query_cat_2", "query_cat_3", "query_cat_4"]
TARGET_CATS = ["target_cat_2", "target_cat_3", "target_cat_4"]
PAIR_COLS = ["asin_query", "asin_target"] + QUERY_CATS + TARGET_CATS

item_cats = df_features[["asin"] + CAT_LEVELS]
mapping = comp_cat[SRC_COLS + DST_COLS].astype(str)


def build_tower_pairs(pairs):
    """Co-purchase pairs -> directed (query, target) rows the mapping licenses."""
    with_cats = (
        pairs
        .assign(asinA=pairs["asinA"].astype(str), asinB=pairs["asinB"].astype(str))
        .merge(item_cats.add_prefix("a_"), left_on="asinA", right_on="a_asin", how="inner")
        .merge(item_cats.add_prefix("b_"), left_on="asinB", right_on="b_asin", how="inner")
        .drop(columns=["a_asin", "b_asin"])
    )
    a_cats = [f"a_{c}" for c in CAT_LEVELS]
    b_cats = [f"b_{c}" for c in CAT_LEVELS]

    def directed(query_asin, query_cats, target_asin, target_cats):
        out = with_cats.merge(mapping, left_on=query_cats + target_cats,
                              right_on=SRC_COLS + DST_COLS, how="inner")
        return out.rename(columns=dict(
            [(query_asin, "asin_query"), (target_asin, "asin_target")]
            + list(zip(query_cats, QUERY_CATS))
            + list(zip(target_cats, TARGET_CATS))))[PAIR_COLS]

    both = pd.concat([directed("asinA", a_cats, "asinB", b_cats),
                      directed("asinB", b_cats, "asinA", a_cats)], ignore_index=True)
    return both, len(with_cats)


slices = {}
for name, pairs in co_pairs.items():
    slices[name], joined = build_tower_pairs(pairs)
    print(f"{name:<6} {len(pairs):>11,} pairs -> {joined:>11,} with features both ends "
          f"({joined / len(pairs):>5.1%}) -> {len(slices[name]):>10,} directed rows")

held_out = len(slices["test"]) / sum(len(v) for v in slices.values())
print(f"\nheld out: {held_out:.1%}")
slices["train"].head(5)

train   10,595,885 pairs ->   7,729,936 with features both ends (73.0%) ->  2,916,635 directed rows
test       942,795 pairs ->     696,070 with features both ends (73.8%) ->    270,957 directed rows

held out: 8.5%


,asin_query,asin_target,query_cat_2,query_cat_3,query_cat_4,target_cat_2,target_cat_3,target_cat_4
0,0560467893,B007EAROSK,Home Dcor,Home Dcor Accents,Corner Shelves,Bath,Bathroom Accessories,Holders & Dispensers
1,0560467893,B0150ZOXEI,Home Dcor,Home Dcor Accents,Corner Shelves,Bath,Bathroom Accessories,Holders & Dispensers
2,0681795107,B000Z4ETF8,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware
3,0681795107,B003ZYGQVK,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,Kitchen & Dining,Cookware,Canning
4,0681795107,B00X5ETKU4,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,Kitchen & Dining,"Coffee, Tea & Espresso",Coffee Makers


## 5. Fold rare brands — **fitted on train**

`brand_clean` folds brands carried by ten or fewer items into `other_brands`,
taking 98,532 brands down to roughly 12,700.

The threshold is counted over **items that appear in the training pairs only**.
Counting over the whole catalogue would let the test period decide which brands
are common enough to keep, which is exactly the kind of quiet leak that is hard
to find later. The resulting fold is then applied to every item, test included —
fit once on train, apply to both.


In [83]:
# Fold rare brands: keep those carried by MORE THAN 10 distinct items. A brand
# on three items would get an embedding trained by a handful of gradient
# updates; bucketing those into one `other_brands` symbol is more honest.
MIN_ITEMS = 10
OTHER = "other_brands"

# FITTED ON TRAIN. The count is taken over items appearing in the training
# pairs, so the test period has no say in which brands survive.
train_items = pd.unique(pd.concat(
    [slices["train"]["asin_query"], slices["train"]["asin_target"]], ignore_index=True))

# Count on brand_norm where the pipeline produced it, so "3d rose" and "3drose"
# are not counted separately and pushed under the threshold by a split spelling.
source = "brand_norm" if "brand_norm" in df_features.columns else "brand"
train_features = df_features[df_features["asin"].isin(train_items)]
brand_counts = train_features.groupby(source)["asin"].nunique()
kept_brands = brand_counts[brand_counts > MIN_ITEMS].index

# APPLIED TO ALL ITEMS, so test rows are folded by the training rule.
df_features["brand_clean"] = df_features[source].where(
    df_features[source].isin(kept_brands) | df_features[source].isna(), OTHER)

print(f"counted on   : {source}, over {len(train_items):,} training items")
print(f"brands kept  : {len(kept_brands):,} of "
      f"{train_features[source].nunique():,} seen in training")
print(f"all items    : {(df_features['brand_clean'] == OTHER).mean():.1%} in {OTHER}, "
      f"{df_features[source].isna().mean():.1%} missing (left as NaN for §7)")
print()
print(df_features["brand_clean"].value_counts().head(10))

counted on   : brand, over 146,984 training items
brands kept  : 2,322 of 25,726 seen in training
all items    : 50.9% in other_brands, 5.7% missing (left as NaN for §7)

brand_clean
other_brands     577792
3dRose             8569
CafePress          7489
Disney             6249
Unknown            5594
Hallmark           4720
Generic            4500
Department 56      3416
Safavieh           3227
Kurt Adler         3124
Name: count, dtype: int64


## 6. Attach the item attributes, and fit the vocabularies

Each side gains seven columns — the item's own content, which is what the
product encoder reads. Names are lowercase throughout:

| Column | Source | Coverage on paired items |
| --- | --- | --- |
| `*_title_cleaned` | `title_cleaned` | 100% |
| `*_price` | `price`, parsed to a number | 75% |
| `*_brand_clean` | `brand_clean` from §5 | 99% |
| `*_product_type` | `Product_Type` | 76.5% |
| `*_material` | `Material` | 72.5% |
| `*_features` | `Features` | 61.7% |
| `*_color` | `Color` | 54.2% |

**`price` needs parsing, not just carrying.** It is stored as a string
(`'$37.00'`), and 7,032 of its non-null values are not prices at all but scraped
CSS. Stripping the currency symbol and coercing handles both: real prices become
floats, the junk becomes `NaN`.

### The vocabularies are the fitted part

Every categorical — the three category levels plus the five string attributes —
gets **one `CategoricalDtype` fitted on the training slice** and applied to
both. Two properties depend on this and both are silent failures if got wrong:

- **Across towers.** A value must index the same embedding row whether it
  arrives as a query or a target. Casting each column independently gives them
  different category lists, which is what made `query_cat_3 == target_cat_3`
  raise earlier — and would have had `Clocks` at code 1 in one tower and code 0
  in the other.
- **Across slices.** The same applies between train and test. Fitting the
  vocabulary twice would leave a category on a different code at evaluation
  time, so the model would look up the wrong vector for a value it knows
  perfectly well.

A test value outside the fitted vocabulary becomes `NaN` on cast and is filled
in §7. That is rare here — 236 rows in 541,914, about 0.04% — and §7 prints the
count so it cannot start growing unnoticed.


In [84]:
# --- Item attributes, one row per asin ------------------------------------
# Source column in df_features -> the name it takes in the pair tables.
ITEM_ATTRS = {
    "title_cleaned": "title_cleaned",
    "price": "price",
    "brand_clean": "brand_clean",
    "Color": "color",
    "Features": "features",
    "Material": "material",
    "Product_Type": "product_type",
}
CATEGORICAL_ATTRS = ["cat_2", "cat_3", "cat_4", "brand_clean",
                     "color", "features", "material", "product_type"]
SIDE_COLS = ["asin_{s}", "{s}_cat_2", "{s}_cat_3", "{s}_cat_4",
             "{s}_title_cleaned", "{s}_price", "{s}_brand_clean",
             "{s}_color", "{s}_features", "{s}_material", "{s}_product_type"]

item_attrs = (df_features[["asin"] + list(ITEM_ATTRS)].rename(columns=ITEM_ATTRS))
item_attrs["price"] = pd.to_numeric(
    item_attrs["price"].astype(str).str.replace(r"[$,]", "", regex=True),
    errors="coerce")
attrs_by_asin = item_attrs.set_index("asin")
print(f"price parsed : {item_attrs['price'].notna().sum():,} of {len(item_attrs):,} "
      f"items ({item_attrs['price'].notna().mean():.1%})")


def attach_attributes(frame):
    """Item content for both sides. `.map`, not `merge`, so re-running is safe."""
    for side in ("query", "target"):
        asin = frame[f"asin_{side}"]
        for attr in ITEM_ATTRS.values():
            frame[f"{side}_{attr}"] = asin.map(attrs_by_asin[attr])
    return frame[[c.format(s=s) for s in ("query", "target") for c in SIDE_COLS]]


for name in slices:
    slices[name] = attach_attributes(slices[name])

# --- Fit the vocabularies on TRAIN, apply to both -------------------------
# `title_cleaned` is excluded: it is free text for the encoder, not a symbol to
# look up, and 145k levels would only cost memory.
vocabularies = {}
for attr in CATEGORICAL_ATTRS:
    values = set()
    for side in ("query", "target"):
        values |= set(slices["train"][f"{side}_{attr}"].dropna().astype(str))
    vocabularies[attr] = pd.CategoricalDtype(sorted(values))

print(f"\nvocabularies fitted on {len(slices['train']):,} training rows:")
for attr, dtype in vocabularies.items():
    print(f"  {attr:<14} {len(dtype.categories):>8,} levels")
slices["train"].head(5)

price parsed : 515,980 of 1,134,566 items (45.5%)

vocabularies fitted on 2,916,635 training rows:
  cat_2                 7 levels
  cat_3                69 levels
  cat_4               395 levels
  brand_clean       2,323 levels
  color                63 levels
  features            542 levels
  material            211 levels
  product_type      1,106 levels


,asin_query,query_cat_2,query_cat_3,query_cat_4,query_title_cleaned,query_price,query_brand_clean,query_color,query_features,query_material,query_product_type,asin_target,target_cat_2,target_cat_3,target_cat_4,target_title_cleaned,target_price,target_brand_clean,target_color,target_features,target_material,target_product_type
0,0560467893,Home Dcor,Home Dcor Accents,Corner Shelves,welland chicago wall floating corner shelf 20 ...,NaN,WELLAND,black,floating shelf,None,corner shelf,B007EAROSK,Bath,Bathroom Accessories,Holders & Dispensers,simplehuman precision lever square push soap p...,15.99,simplehuman,None,None,plastic,soap pump
1,0560467893,Home Dcor,Home Dcor Accents,Corner Shelves,welland chicago wall floating corner shelf 20 ...,NaN,WELLAND,black,floating shelf,None,corner shelf,B0150ZOXEI,Bath,Bathroom Accessories,Holders & Dispensers,double toothbrush holder angle simple sus304 s...,22.99,other_brands,None,wall mounted,stainless steel,toothbrush holder
2,0681795107,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,stainless coffee mug,14.27,Timolino,None,None,stainless,mug,B000Z4ETF8,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware,tervis 1001833 clear colorful insulated tumble...,24.00,other_brands,clear,insulated,melamine,tumbler
3,0681795107,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,stainless coffee mug,14.27,Timolino,None,None,stainless,mug,B003ZYGQVK,Kitchen & Dining,Cookware,Canning,tervis travel lid 24 oz orange,7.37,Tervis,orange,dishwasher safe,None,lid
4,0681795107,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,stainless coffee mug,14.27,Timolino,None,None,stainless,mug,B00X5ETKU4,Kitchen & Dining,"Coffee, Tea & Espresso",Coffee Makers,mr coffee 12-cup programmable coffee maker the...,NaN,Mr. Coffee,None,carafe,chrome,coffee maker


## 7. Fill the missing values — **fitted on train**

### Prices — the category median

About a quarter of rows have no price. Each takes the median price of its own
category, from a table fitted on the training period alone.

**The population is items reviewed before the cutoff**, not the whole
catalogue. Two consequences, and the first is why:

- *Temporally clean.* Nothing that happens at or after `date_threshold` can
  influence a training feature. On this data the effect is small — only 277
  items are test-period-only, moving 2 of 697 medians by more than $1 — but it
  is the difference between a guarantee and a measurement that has to be
  re-checked whenever the split moves.
- *Smaller and noisier.* Restricting from ~1.29M catalogue items to the ~189k
  that were actually reviewed cuts the sample behind each median roughly in
  half and leaves some paths unpriced. The **fallback ladder**
  `cat_4 → cat_3 → cat_2 → global` closes those.

`*_price_imputed` records which values were inferred, because a quarter of the
column is being filled and whether a price is real is itself a signal.

### Categoricals — the fitted vocabulary, then an explicit `Missing`

Applying §6's train-fitted `CategoricalDtype` does two jobs at once. Values in
the vocabulary keep a stable code across both towers and both slices. Values
outside it — only possible in test — become `NaN` on cast and are then filled
with `Missing`, alongside the genuinely absent ones.

Folding unseen values into `Missing` rather than a separate `Unknown` is
deliberate. `Unknown` would carry ~236 training examples, so its embedding would
never leave its initialisation and whatever the model did with it would be
arbitrary. `Missing` is well trained and means "no usable value for this
attribute", which is a fair description of the situation. The cell prints the
unseen count so that if it ever grows past a fraction of a percent, the decision
can be revisited.

⚠️ Every statistic in this section is fitted on `slices["train"]` and applied to
both. Nothing is recomputed on the test slice.


In [85]:
from complementary_cats_pairs import fold_cat_4, parse_category_levels

# --- Category price medians: FITTED ON TRAIN ------------------------------
# Population: items reviewed strictly before the cutoff. The category path is
# parsed and folded exactly as §3 folds df_features, so the medians key onto
# query_cat_2/3/4 and target_cat_2/3/4 directly.
train_reviewed = set(df_reviews.loc[df_reviews["unixReviewTime"] < cutoff_time, "asin"])

cat_levels = parse_category_levels(meta_catalogue["category"], n_levels=4)
catalogue_prices = pd.DataFrame({
    "asin": meta_catalogue["asin"],
    "cat_2": cat_levels["cat_2"].astype(str),
    "cat_3": cat_levels["cat_3"].astype(str),
    "cat_4": fold_cat_4(cat_levels["cat_3"], cat_levels["cat_4"], valid_pairs),
    "price": pd.to_numeric(
        meta_catalogue["price"].astype(str).str.replace(r"[$,]", "", regex=True),
        errors="coerce"),
})
fit_prices = catalogue_prices[catalogue_prices["asin"].isin(train_reviewed)]

median_4 = fit_prices.groupby(["cat_2", "cat_3", "cat_4"], observed=True)["price"].median()
median_3 = fit_prices.groupby(["cat_2", "cat_3"], observed=True)["price"].median()
median_2 = fit_prices.groupby(["cat_2"], observed=True)["price"].median()
median_all = fit_prices["price"].median()

print(f"median population : {len(fit_prices):,} items reviewed before "
      f"{DATE_THRESHOLD} ({fit_prices['price'].notna().sum():,} priced)")
print(f"median tables     : {median_4.notna().sum():,} cat_4 paths | "
      f"{median_3.notna().sum()} cat_3 | {median_2.notna().sum()} cat_2 | "
      f"global ${median_all:,.2f}")


def fill_prices(frame):
    """Fill each missing price from its category median, falling back up."""
    for side in ("query", "target"):
        raw = frame[f"asin_{side}"].map(attrs_by_asin["price"])
        c2 = frame[f"{side}_cat_2"].astype(str)
        c3 = frame[f"{side}_cat_3"].astype(str)
        c4 = frame[f"{side}_cat_4"].astype(str)
        at_4 = pd.Series(median_4.reindex(pd.MultiIndex.from_arrays([c2, c3, c4])).to_numpy(),
                         index=frame.index)
        at_3 = pd.Series(median_3.reindex(pd.MultiIndex.from_arrays([c2, c3])).to_numpy(),
                         index=frame.index)
        at_2 = pd.Series(median_2.reindex(pd.Index(c2)).to_numpy(), index=frame.index)
        missing = raw.isna()
        frame[f"{side}_price"] = raw.fillna(
            at_4.fillna(at_3).fillna(at_2).fillna(median_all))
        frame[f"{side}_price_imputed"] = missing
        yield side, missing, at_4, at_3, at_2


def apply_vocabularies(frame):
    """Cast to the train-fitted dtypes; unseen values fall out as NaN."""
    unseen = {}
    for attr, dtype in vocabularies.items():
        for side in ("query", "target"):
            col = f"{side}_{attr}"
            values = frame[col].astype(object)
            cast = pd.Categorical(values, dtype=dtype)
            unseen[attr] = unseen.get(attr, 0) + int(
                (values.notna() & pd.isna(cast)).sum())
            frame[col] = cast
    return unseen


for name, frame in slices.items():
    print(f"\n--- {name} ---")
    for side, missing, at_4, at_3, at_2 in fill_prices(frame):
        rungs = (int((missing & at_4.notna()).sum()),
                 int((missing & at_4.isna() & at_3.notna()).sum()),
                 int((missing & at_4.isna() & at_3.isna() & at_2.notna()).sum()),
                 int((missing & at_4.isna() & at_3.isna() & at_2.isna()).sum()))
        print(f"  {side:<7} {missing.sum():>9,} missing ({missing.mean():>5.1%}) -> "
              f"cat_4 {rungs[0]:,} | cat_3 {rungs[1]:,} | cat_2 {rungs[2]:,} | "
              f"global {rungs[3]:,} | still NaN "
              f"{frame[f'{side}_price'].isna().sum():,}")
    unseen = apply_vocabularies(frame)
    total_unseen = sum(unseen.values())
    print(f"  values outside the training vocabulary: {total_unseen:,} "
          f"({total_unseen / (len(frame) * 2 * len(vocabularies)):.4%} of slots)"
          + (f"  {ute}" if (ute := {k: v for k, v in unseen.items() if v}) else ""))

median population : 193,265 items reviewed before 2017-12-09 (120,916 priced)
median tables     : 584 cat_4 paths | 159 cat_3 | 13 cat_2 | global $16.45

--- train ---
  query     726,061 missing (24.9%) -> cat_4 726,061 | cat_3 0 | cat_2 0 | global 0 | still NaN 0
  target    721,768 missing (24.7%) -> cat_4 721,768 | cat_3 0 | cat_2 0 | global 0 | still NaN 0
  values outside the training vocabulary: 0 (0.0000% of slots)

--- test ---
  query      38,727 missing (14.3%) -> cat_4 38,727 | cat_3 0 | cat_2 0 | global 0 | still NaN 0
  target     38,301 missing (14.1%) -> cat_4 38,301 | cat_3 0 | cat_2 0 | global 0 | still NaN 0
  values outside the training vocabulary: 6 (0.0001% of slots)  {'features': 4, 'product_type': 2}


In [86]:
MISSING_LABEL = "Missing"


def fill_missing_categories(frame, attrs, label=MISSING_LABEL):
    """Give every missing categorical value an explicit `label` level.

    Covers both the genuinely absent and anything that fell outside the
    train-fitted vocabulary in the cell above -- after the cast both are NaN,
    and both mean the same thing to the encoder: no usable value here.

    A pandas Categorical refuses a value that is not one of its categories, so
    the level is added before it is assigned. Idempotent.
    """
    for attr in attrs:
        for side in ("query", "target"):
            col = f"{side}_{attr}"
            series = frame[col]
            if label not in series.cat.categories:
                series = series.cat.add_categories([label])
            frame[col] = series.fillna(label)
    return frame


for name, frame in slices.items():
    before = {c: frame[c].isna().sum()
              for c in (f"{s}_{a}" for s in ("query", "target") for a in CATEGORICAL_ATTRS)}
    slices[name] = fill_missing_categories(frame, CATEGORICAL_ATTRS)
    filled = sum(before.values())
    print(f"{name:<6} filled {filled:>9,} categorical values with {MISSING_LABEL!r} "
          f"({filled / (len(frame) * 2 * len(CATEGORICAL_ATTRS)):.2%} of slots)")

print("\nvocabularies identical across towers and slices:")
for attr in CATEGORICAL_ATTRS:
    levels = [list(slices[n][f"{s}_{attr}"].cat.categories)
              for n in slices for s in ("query", "target")]
    print(f"  {attr:<14} {all(l == levels[0] for l in levels)}  "
          f"({len(levels[0]):,} levels, {MISSING_LABEL!r} present: "
          f"{MISSING_LABEL in levels[0]})")

train  filled 7,096,407 categorical values with 'Missing' (15.21% of slots)
test   filled   589,876 categorical values with 'Missing' (13.61% of slots)

vocabularies identical across towers and slices:
  cat_2          True  (8 levels, 'Missing' present: True)
  cat_3          True  (70 levels, 'Missing' present: True)
  cat_4          True  (395 levels, 'Missing' present: True)
  brand_clean    True  (2,324 levels, 'Missing' present: True)
  color          True  (64 levels, 'Missing' present: True)
  features       True  (543 levels, 'Missing' present: True)
  material       True  (212 levels, 'Missing' present: True)
  product_type   True  (1,107 levels, 'Missing' present: True)


## 8. `target_node` — the target category as one symbol

The three target levels collapsed into a single value, e.g.
`Bath > Bathroom Accessories > Holders & Dispensers`.

This is the **target category** the paper's query tower consumes: in
Complementary-TT the query product embedding is concatenated with a target
category embedding, and that embedding needs one symbol per category to look
up, not three separate levels. Keeping the full path rather than `cat_4` alone
matters because a leaf label is not unique on its own — `Missing` and
`<cat_3>_Other` recur under many parents, so `cat_4` by itself would collapse
genuinely different categories onto one embedding row.

Built **after** §7, deliberately: every level is complete by then, so no node
carries a `NaN` fragment or silently becomes the string `"nan"`.

Its vocabulary is **fitted on train** like every other, and a test node outside
it falls back to `Missing` — the same rule §7 applies to the individual levels.
Because §4 only keeps pairs whose category path is in the mapping, that should
never fire; the cell asserts it rather than assuming.

Finally the two slices are named `tower_pairs_train` and `tower_pairs_test`.
There is no bare `tower_pairs` — with two slices in play the unqualified name
would be ambiguous about which one it meant.

### Dropping same-category pairs

A pair whose query and target sit in the same `cat_4` is a substitute, not a
complement — a table recommended for a table. Those rows are removed from both
slices.

They exist because `categories.ipynb` keeps the mapping's 391 self-pairs on
purpose ("a chair listed with another chair is a real `also_buy` edge. Filter
them out downstream if the use needs strict complements"), so this is that
downstream filter.

`cat_4` alone would normally be an unsafe test — leaf labels recur under
different parents, and 11,784 rows have a matching `cat_4` beneath a *different*
`cat_3`. Excluding `Missing` removes almost all of those: with the exclusion the
`cat_4` rule and a full-path comparison agree to within 750 rows out of 2.9M.

| | train | test |
| --- | --- | --- |
| removed (`cat_4` equal, not `Missing`) | 692,640 (23.7%) | 69,760 (25.7%) |
| kept back (`cat_4` equal, both `Missing`) | 61,628 (2.1%) | 5,499 (2.0%) |

The second row is what this rule deliberately leaves in: pairs where neither
item has a `cat_4`, most of which do share a `cat_2` and `cat_3`. Widening to a
full-path comparison would remove those too — a further 1.8% — if the
substitutes they contain turn out to matter.


In [87]:
# One symbol per target category path, for the query tower's target-category
# embedding. `" > "` is cosmetic -- it only has to be readable and not appear
# inside a category name.
NODE_SEP = " > "
TARGET_LEVELS = ["target_cat_2", "target_cat_3", "target_cat_4"]


def target_node(frame):
    return (frame["target_cat_2"].astype(str) + NODE_SEP
            + frame["target_cat_3"].astype(str) + NODE_SEP
            + frame["target_cat_4"].astype(str))


# FITTED ON TRAIN, applied to both.
node_vocabulary = pd.CategoricalDtype(sorted(set(target_node(slices["train"]))))
print(f"target_node vocabulary: {len(node_vocabulary.categories):,} paths, "
      f"fitted on {len(slices['train']):,} training rows")

for name, frame in slices.items():
    nodes = target_node(frame)
    cast = pd.Categorical(nodes, dtype=node_vocabulary)
    unseen = int(pd.isna(cast).sum())
    if unseen:                     # never expected: §4 filters on the mapping
        cast = cast.add_categories([MISSING_LABEL]).fillna(MISSING_LABEL)
    frame["target_node"] = cast
    print(f"  {name:<6} {nodes.nunique():>4,} distinct | outside the training "
          f"vocabulary: {unseen:,}")

for name, frame in slices.items():
    print(f"\n{name:<6} {str(frame.shape):>16} | "
          f"{frame.memory_usage(deep=True).sum() / 1e6:>6,.0f} MB | "
          f"price complete: "
          f"{frame['query_price'].notna().all() and frame['target_price'].notna().all()}")
print(f"\ncolumns ({slices['train'].shape[1]}): {list(slices['train'].columns)}")
slices["train"].head(5)

target_node vocabulary: 448 paths, fitted on 2,916,635 training rows
  train   448 distinct | outside the training vocabulary: 0
  test    425 distinct | outside the training vocabulary: 0

train     (2916635, 25) |  1,202 MB | price complete: True

test       (270957, 25) |    117 MB | price complete: True

columns (25): ['asin_query', 'query_cat_2', 'query_cat_3', 'query_cat_4', 'query_title_cleaned', 'query_price', 'query_brand_clean', 'query_color', 'query_features', 'query_material', 'query_product_type', 'asin_target', 'target_cat_2', 'target_cat_3', 'target_cat_4', 'target_title_cleaned', 'target_price', 'target_brand_clean', 'target_color', 'target_features', 'target_material', 'target_product_type', 'query_price_imputed', 'target_price_imputed', 'target_node']


,asin_query,query_cat_2,query_cat_3,query_cat_4,query_title_cleaned,query_price,query_brand_clean,query_color,query_features,query_material,query_product_type,asin_target,target_cat_2,target_cat_3,target_cat_4,target_title_cleaned,target_price,target_brand_clean,target_color,target_features,target_material,target_product_type,query_price_imputed,target_price_imputed,target_node
0,0560467893,Home Dcor,Home Dcor Accents,Corner Shelves,welland chicago wall floating corner shelf 20 ...,18.24,WELLAND,black,floating shelf,Missing,corner shelf,B007EAROSK,Bath,Bathroom Accessories,Holders & Dispensers,simplehuman precision lever square push soap p...,15.99,simplehuman,Missing,Missing,plastic,soap pump,True,False,Bath > Bathroom Accessories > Holders & Dispen...
1,0560467893,Home Dcor,Home Dcor Accents,Corner Shelves,welland chicago wall floating corner shelf 20 ...,18.24,WELLAND,black,floating shelf,Missing,corner shelf,B0150ZOXEI,Bath,Bathroom Accessories,Holders & Dispensers,double toothbrush holder angle simple sus304 s...,22.99,other_brands,Missing,wall mounted,stainless steel,toothbrush holder,True,False,Bath > Bathroom Accessories > Holders & Dispen...
2,0681795107,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,stainless coffee mug,14.27,Timolino,Missing,Missing,stainless,mug,B000Z4ETF8,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware,tervis 1001833 clear colorful insulated tumble...,24.00,other_brands,clear,insulated,melamine,tumbler,False,False,Kitchen & Dining > Dining & Entertaining > Gla...
3,0681795107,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,stainless coffee mug,14.27,Timolino,Missing,Missing,stainless,mug,B003ZYGQVK,Kitchen & Dining,Cookware,Canning,tervis travel lid 24 oz orange,7.37,Tervis,orange,dishwasher safe,Missing,lid,False,False,Kitchen & Dining > Cookware > Canning
4,0681795107,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,stainless coffee mug,14.27,Timolino,Missing,Missing,stainless,mug,B00X5ETKU4,Kitchen & Dining,"Coffee, Tea & Espresso",Coffee Makers,mr coffee 12-cup programmable coffee maker the...,16.99,Mr. Coffee,Missing,carafe,chrome,coffee maker,False,True,"Kitchen & Dining > Coffee, Tea & Espresso > Co..."


In [88]:
# --- Drop same-category pairs: substitutes, not complements ---------------
# Applied to both slices with the same rule -- this is a per-row predicate,
# not a fitted statistic, so it carries no train/test asymmetry.
def drop_same_category(frame):
    same = ((frame["query_cat_4"] == frame["target_cat_4"])
            & (frame["query_cat_4"] != MISSING_LABEL))
    return frame.loc[~same].reset_index(drop=True), int(same.sum())


for name in slices:
    before = len(slices[name])
    slices[name], removed = drop_same_category(slices[name])
    print(f"{name:<6} {before:>10,} -> {len(slices[name]):>10,} rows "
          f"(dropped {removed:,}, {removed / before:.1%})")

tower_pairs_train = slices["train"]
tower_pairs_test = slices["test"]
print(f"\nheld out: "
      f"{len(tower_pairs_test) / (len(tower_pairs_train) + len(tower_pairs_test)):.1%}")

# The node vocabulary was fitted before this filter, so some of its 448 paths
# may no longer occur. Harmless -- an unused embedding row is never updated --
# but worth seeing rather than assuming.
still_used = tower_pairs_train["target_node"].nunique()
print(f"target_node paths still present in train: {still_used:,} of "
      f"{len(node_vocabulary.categories):,}")

train   2,916,635 ->  2,223,995 rows (dropped 692,640, 23.7%)
test      270,957 ->    201,197 rows (dropped 69,760, 25.7%)

held out: 8.3%
target_node paths still present in train: 432 of 448


## 9. Export the model-ready arrays

Turns the two tables into what a training loop wants: **pairs hold integer
indices only, and every feature lives once per item.** A pair table repeating a
384-dim title vector on both sides would be 2.9M × 768 floats — 9 GB — to say
the same thing 160k rows of `items.npz` say.

Written to `data/tower/`:

| File | Contents |
| --- | --- |
| `items.npz` | `title_emb` (n × 384), `cat_ids` (n × 8), `numeric` (n × 3) |
| `pairs_train.parquet`, `pairs_test.parquet` | `query_idx`, `target_idx`, `target_node_id`, `weight` |
| `vocabs.json` | every categorical vocabulary, so training and serving agree |
| `node_of_item.npy` | each item's own target-category id |

**Vocabulary id 0 is reserved** as the padding/unseen index, so every real value
starts at 1. §7 already folded unseen values into `Missing`, so 0 should never
be produced here — the cell asserts that rather than trusting it.

### Three fixes against the draft script

- **`price` was read from `price_imputed`.** That column is the boolean flag
  §7 sets, not a price, so `log1p` was being taken of 0/1 and the actual price
  discarded. The numeric block now reads `price` and uses `price_imputed` as the
  missingness indicator — which also restores the signal, since `price.isna()`
  is uniformly `False` after §7 and would have made that feature a column of
  zeros.
- **`node_of_item` was scattered from the training pairs**, leaving every item
  that never appears as a *target* at id 0. A node is a function of the item's
  own category path, so it is derived directly from `items` and is complete.
- **The price decile was divided by 10** while `duplicates="drop"` can return
  fewer than ten bins, so the feature was not on `[0, 1]`. It is normalised by
  the bin count actually produced.

Titles reuse the 384-dim SBERT vectors already in
`df_features_with_embeddings.pkl` rather than re-encoding — same model, and it
is the vector `embedding_analysis/` produced. Set `ENCODE_TITLES = True` to
compute them instead (~2.5 min on CPU); the precomputed path costs a transient
4.8 GB while the frame is open.


In [89]:
from pathlib import Path

OUT_DIR = DATA_DIR / "tower"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ENCODE_TITLES = False          # True -> run SBERT instead of reusing the pickle

CAT_ORDER = ["cat_2", "cat_3", "cat_4", "brand", "color", "material",
             "product_type", "features"]
# column in the pair tables (without the query_/target_ prefix) -> canonical name
COLMAP = {"cat_2": "cat_2", "cat_3": "cat_3", "cat_4": "cat_4",
          "brand_clean": "brand", "color": "color", "material": "material",
          "product_type": "product_type", "features": "features"}


def _side(df, prefix):
    cols = {f"{prefix}_{src}": dst for src, dst in COLMAP.items()}
    cols[f"asin_{prefix}"] = "asin"
    cols[f"{prefix}_title_cleaned"] = "title"
    cols[f"{prefix}_price"] = "price"
    cols[f"{prefix}_price_imputed"] = "price_imputed"
    return df[list(cols)].rename(columns=cols)


def build_items(train, test):
    """One row per asin, from every side of both slices."""
    frames = [_side(d, p) for d in (train, test) for p in ("query", "target")]
    items = (pd.concat(frames, ignore_index=True)
               .drop_duplicates("asin").reset_index(drop=True))
    items["idx"] = np.arange(len(items))
    return items


def fit_vocabs(train):
    """FIT ON TRAIN ONLY. Id 0 is reserved for padding / unseen."""
    vocabs = {}
    for src, dst in COLMAP.items():
        values = pd.unique(pd.concat(
            [train[f"query_{src}"].astype(str), train[f"target_{src}"].astype(str)],
            ignore_index=True))
        vocabs[dst] = {v: i + 1 for i, v in enumerate(sorted(values))}
    nodes = sorted(train["target_node"].astype(str).unique())
    vocabs["target_node"] = {v: i + 1 for i, v in enumerate(nodes)}
    return vocabs


def encode_items(items, vocabs, title_emb):
    cat_ids = np.zeros((len(items), len(CAT_ORDER)), dtype=np.int64)
    for j, name in enumerate(CAT_ORDER):
        cat_ids[:, j] = items[name].astype(str).map(vocabs[name]).fillna(0).to_numpy()

    # FIX: the price, not the imputation flag. `price_imputed` is the indicator.
    price = items["price"].astype(float)
    imputed = items["price_imputed"].astype("float32")
    price = price.fillna(price.median())          # defensive; §7 leaves none

    # FIX: normalise by the bins qcut actually produced, not a hardcoded 10.
    decile = pd.qcut(price, 10, labels=False, duplicates="drop")
    decile = decile.fillna(decile.median())
    numeric = np.stack([
        np.log1p(price.to_numpy()).astype("float32"),
        (decile.to_numpy() / max(decile.max(), 1)).astype("float32"),
        imputed.to_numpy(),
    ], axis=1).astype("float32")
    return {"title_emb": title_emb.astype("float32"),
            "cat_ids": cat_ids, "numeric": numeric}


def slim_pairs(df, items, vocabs):
    idx = dict(zip(items["asin"], items["idx"]))
    out = pd.DataFrame({
        "query_idx": df["asin_query"].map(idx),
        "target_idx": df["asin_target"].map(idx),
        "target_node_id": df["target_node"].astype(str)
                            .map(vocabs["target_node"]).fillna(0).astype(int),
        "weight": df.get("weight", pd.Series(1.0, index=df.index)).astype("float32"),
    }).dropna()
    return out.astype({"query_idx": int, "target_idx": int})


def item_title_embeddings(items):
    if ENCODE_TITLES:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer("all-MiniLM-L6-v2")
        return np.asarray(model.encode(items["title"].astype(str).fillna("").tolist(),
                                       batch_size=256, show_progress_bar=True))
    # Reuse the vectors embedding_analysis/ already produced (384-d, unit-norm).
    # Narrowed to our asins before anything is stacked: the pickle holds 1.13M
    # vectors and we need ~160k, so a lookup over the whole frame is the
    # difference between seconds and minutes.
    wanted = set(items["asin"])
    frame = pd.read_pickle(DATA_DIR / "df_features_with_embeddings.pkl")[
        ["asin", "title_embedding"]]
    frame = frame[frame["asin"].isin(wanted)]
    vectors = frame.set_index("asin")["title_embedding"].reindex(items["asin"])
    del frame

    found = vectors.notna().to_numpy()
    dim = len(vectors[found].iloc[0])
    out = np.zeros((len(items), dim), dtype="float32")
    out[found] = np.stack(vectors[found].to_numpy()).astype("float32")
    print(f"title vectors: {int(found.sum()):,} matched, "
          f"{int((~found).sum()):,} missing (left as zeros)")
    return out


# --------------------------------------------------------------------------
items = build_items(tower_pairs_train, tower_pairs_test)
vocabs = fit_vocabs(tower_pairs_train)               # TRAIN ONLY
print(f"items: {len(items):,} unique asins across both slices")

title_emb = item_title_embeddings(items)
arrays = encode_items(items, vocabs, title_emb)

pairs_train = slim_pairs(tower_pairs_train, items, vocabs)
pairs_test = slim_pairs(tower_pairs_test, items, vocabs)

# FIX: a node is a function of the item's own path, so derive it for every
# item. Scattering from the training pairs left query-only items at 0.
item_node = (items["cat_2"].astype(str) + NODE_SEP + items["cat_3"].astype(str)
             + NODE_SEP + items["cat_4"].astype(str))
node_of_item = item_node.map(vocabs["target_node"]).fillna(0).astype(np.int64).to_numpy()

np.savez_compressed(OUT_DIR / "items.npz", **arrays)
json.dump(vocabs, open(OUT_DIR / "vocabs.json", "w"))
pairs_train.to_parquet(OUT_DIR / "pairs_train.parquet")
pairs_test.to_parquet(OUT_DIR / "pairs_test.parquet")
np.save(OUT_DIR / "node_of_item.npy", node_of_item)

# Id 0 is the reserved unseen/padding index, and it exists for exactly one
# case: an item that appears only in test carrying a value the train-fitted
# vocabulary never saw. A TRAINING item landing there would be a real defect,
# since the vocabulary was fitted on those very rows.
train_items = np.unique(pairs_train[["query_idx", "target_idx"]].to_numpy())
test_items = np.unique(pairs_test[["query_idx", "target_idx"]].to_numpy())
assert (arrays["cat_ids"][train_items] > 0).all(), \
    "a training item fell on the reserved id 0 — the vocabulary is inconsistent"
assert (pairs_train["target_node_id"] > 0).all(), "a train node fell on id 0"

test_only = np.setdiff1d(test_items, train_items)
zero_slots = int((arrays["cat_ids"][test_only] == 0).sum())
unseen_nodes = int((pairs_test["target_node_id"] == 0).sum())

print(f"\nitems {len(items):,} | train {len(pairs_train):,} | test {len(pairs_test):,}")
print(f"title_emb {arrays['title_emb'].shape} | cat_ids {arrays['cat_ids'].shape} "
      f"| numeric {arrays['numeric'].shape}")
print(f"vocab sizes: {({k: len(v) for k, v in vocabs.items()})}")
print(f"test-only items                              : {len(test_only):,}")
print(f"  their categorical slots on the reserved id 0: {zero_slots:,} of "
      f"{len(test_only) * len(CAT_ORDER):,}")
print(f"items with no node in the training vocabulary : {(node_of_item == 0).sum():,}")
print(f"test pairs whose target node is unseen       : {unseen_nodes:,}")
print(f"\nwritten to {OUT_DIR.relative_to(ROOT)}/")
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name:<24} {f.stat().st_size / 1e6:>8,.1f} MB")

items: 137,362 unique asins across both slices
title vectors: 137,362 matched, 0 missing (left as zeros)

items 137,362 | train 2,223,995 | test 201,197
title_emb (137362, 384) | cat_ids (137362, 8) | numeric (137362, 3)
vocab sizes: {'cat_2': 7, 'cat_3': 69, 'cat_4': 386, 'brand': 2311, 'color': 64, 'material': 209, 'product_type': 1099, 'features': 539, 'target_node': 432}
test-only items                              : 2,702
  their categorical slots on the reserved id 0: 3 of 21,616
items with no node in the training vocabulary : 219
test pairs whose target node is unseen       : 0

written to data/tower/
  items.npz                   196.9 MB
  node_of_item.npy              1.1 MB
  pairs_test.parquet            1.1 MB
  pairs_train.parquet           9.2 MB
  ttn_complementary.pt          2.8 MB
  vocabs.json                   0.1 MB


## 10. The two-tower model

Both towers are the paper's **Product Encoder**: an embedding table per
categorical, the 384-dim title vector, the numeric block, concatenated through
an MLP and L2-normalised. They share the architecture, not the weights. Only
the query tower is modified — its product embedding is concatenated with a
**target-category embedding** before the final projection, which is the single
asymmetry Complementary-TT introduces.

Scoring is a dot product of two normalised vectors, i.e. cosine.

### The query tower is told which category to retrieve

At training time `target_node_id` comes from the target item, which is the
label. That is the paper's design rather than a leak — at serving you supply
the desired category from `complementary_categories.pkl` instead of reading it
off the answer. But it does mean these metrics measure **"find the right item
*within* the right category"**, not "find the right category too". Read them
with that in mind.

### Negatives decide whether this works at all

Measured on this data, and it is not a small effect:

| negatives | R@10 | median rank | behaviour |
| --- | --- | --- | --- |
| in-batch only, 300 steps | 0.0445 | 322 | best it gets |
| in-batch only, 2000 steps | 0.0317 | 351 | **degrades** |
| mixed, 500 steps | 0.0876 | 169 | stable |
| mixed, 3000 steps | 0.0954 | 174 | stable |

With in-batch negatives alone the loss keeps falling while retrieval gets
*worse*. A random negative is nearly always from another category, so the model
learns coarse category separation within ~200 steps and then sharpens a proxy
that has stopped tracking the real objective. Adding negatives drawn from the
target's own `target_node` supplies that signal — the paper's "mixed negative
sampling" — and roughly doubles Recall@10.

### Why mined negatives were tried, and why they were reverted

The obvious next step from the table above is to mine negatives from the model's
own current top-K inside the node instead of drawing them uniformly, and it is
what an earlier version of this section proposed. It was implemented and
measured, and **it made the model roughly half as good**. The experiment is kept
in `ttn/experiments/` rather than deleted, because the reason it failed is the
most useful thing this notebook has learned.

The motivation looked solid. On the trained model:

| measurement | value |
| --- | --- |
| P(in-batch negative shares the positive's node) | 0.0113 |
| margin of the in-batch term | 0.846 (already solved) |
| median rank of the uniform in-node negative | 2,211 |
| that negative inside the model's own top-10 | 2.25% |
| `score(top1) - score(top10)`, raw cosine | 0.0067 |

So the head of the ranking is never supervised, and the query tower collapses to
one direction per category: the node branch of `query_out` reaches 13.5x the norm
of the item branch, every query in a node sits within cosine 0.99 of every other,
and 200 different bed frames asking for Mattresses & Box Springs return a pool of
41 items with 7 distinct top-1s.

The ablation (`ttn/experiments/negative_sampling_sweep.log`, 3 epochs each):

| configuration | R@10 | R@100 | median rank |
| --- | --- | --- | --- |
| **this cell, unchanged** | **0.1240** | 0.4559 | 124 |
| + mined negatives (4 mined, 4 uniform) | 0.0636 | 0.3223 | 217 |
| + temperature 0.2 | 0.0810 | 0.3247 | 226 |
| + temperature 0.05 | 0.0659 | 0.2953 | 256 |
| + LayerNorm on the query blocks | 0.0660 | 0.3275 | 213 |

Every variant loses. `ttn/experiments/why_mining_fails.py` shows why:

| negatives drawn per row | mean popularity percentile *within the node* |
| --- | --- |
| mined (hardest 4 of a 256 in-node pool) | 0.836 |
| uniform in-node | 0.500 |
| **the true targets** | **0.832** |

Mining draws negatives from precisely the popularity stratum the correct answers
occupy. It is not finding hard-but-wrong items; it is finding the items that are
usually right, and training the model to push them down. Masking the 2.2M known
`(query, target)` training pairs does not help — only 1.07% of mined negatives
are provably a complement of that query, and the unrecorded ones dominate.

### What the evaluation is really measuring

Two corrections came out of the same work, and both are now in the cell.

**Strict recall understates the model by about half.** A test `(query, category)`
key holds 1.73 held-out targets on average; the ones not being scored sit in the
candidate pool counted as wrong answers. `evaluate(..., lenient=True)` credits
any of them: 0.1179 strict becomes 0.2506 lenient.

**`10 / len(items)` was the wrong baseline.** The query tower is handed the true
target category and 99.6% of its top-10 lands inside it, so the floor is a random
draw *within* that category, and the bar to clear is per-category popularity:

| baseline | R@10 | lenient |
| --- | --- | --- |
| random over the whole catalogue | 0.0001 | — |
| random within the asked-for category | 0.0253 | — |
| **this model** | **0.1179** | **0.2506** |
| **popularity within the category** (one `groupby`) | **0.2136** | **0.4034** |

The model loses to a `groupby`. That is the honest state of it.

### Why, and what it means for the target

`ttn/experiments/query_conditional_signal.log` builds a top-10 purely from
training counts, grouped by progressively more information about the query — no
model at all:

| grouped by | R@10 | lenient |
| --- | --- | --- |
| target category only | 0.2147 | 0.4114 |
| + query `cat_3` | 0.2139 | 0.4128 |
| + query `cat_4` | 0.2167 | 0.4144 |
| + query brand | 0.2099 | 0.4022 |
| + query `product_type` | 0.2039 | 0.3936 |
| + the exact query item | 0.1816 | 0.3377 |

Flat, then declining — but **this table is weaker evidence than it looks, and
§13 revises the conclusion drawn from it.** It is a count-based test, and the raw
`also_buy` field turns out to be so item-specific (Jaccard 0.005 between two
source items pointing at the same category) that there are almost no repeated
query→target observations to count. It therefore cannot distinguish *no signal*
from *signal too sparse to count*, and it should not be read as proof that the
query item is uninformative.

What can be said: after §19's filtering, the surviving target is
popularity-dominated, which is why every loss change above underperformed. The
raw field is not — 36.8% of its edges are same-brand at 161x chance, and the
item-specific half is the same-category half that §19 removes. See §13.

Training walks a **shuffled permutation** each epoch rather than sampling with
replacement. At this length the difference is small, but sampling would leave
about 16% of pairs unseen and make two runs at the same step count disagree on
which data they saw.

### Depth

Each tower is one hidden layer: the attribute embeddings and the title/numeric
projections are concatenated, passed through a single 256-unit ReLU layer, and
projected to 128 before normalising. The query tower adds one further linear
map after the node embedding is concatenated, with no activation. The paper
specifies only "a multi-layer perceptron (FC)", so this is a defensible
minimum rather than a reproduction — and depth is one of the few untried knobs.

### Two implementation notes

`masked_fill`, never `masked_select`, in the loss. `masked_select` has a
data-dependent output size, and on MPS it returned the wrong element count —
`shape '[4096, -1]' is invalid for input of size 18,867,712` when the
off-diagonal of a 4096² matrix has 16,773,120 entries — followed by an
out-of-bounds fault. A fixed-shape mask avoids the whole class of problem.

The numeric block is standardised using **training items only**, consistent
with §§5–8.

Evaluation ranks the true target against the **entire catalogue**, with the
query item itself masked out. A sampled candidate pool would flatter the model;
full-catalogue ranking is the honest measure and is affordable at this size.

### Saving

The trained weights go to `data/tower/ttn_complementary.pt`, together with the
architecture config, the vocabulary sizes and the numeric standardisation
constants. Those travel with the state dict deliberately: every layer's shape is
derived from the §6 vocabularies, so a checkpoint reloaded against a different
fit would mismatch — silently, if only one vocabulary changed size. The cell
reloads the file into a fresh model and asserts it scores identically, so a
restart genuinely resumes rather than merely appearing to.


In [ ]:
import time
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "mps" if torch.backends.mps.is_available() else (
         "cuda" if torch.cuda.is_available() else "cpu")
CAT_DIM, NODE_DIM, HIDDEN, OUT_DIM = 32, 32, 256, 128
BATCH, EPOCHS, LR = 4096, 30, 3e-3
SEED = 0

# Temperature on the cosine scores, and the single most consequential constant
# in this cell. Both towers are L2-normalised, so scores live in [-1, 1] and the
# differences that decide a ranking are tiny: measured on the untempered model,
# score(top1) - score(top10) was 0.0067. At that scale `logsigmoid` is
# effectively linear, so a badly-ordered pair pulls no harder than a well-ordered
# one -- the loss falls while the ranking stays arbitrary.
#
# Isolated on 5,000 pairs inside ONE category (so the node embedding is constant
# and cannot help), 150 epochs of training reached:
#
#     TAU = 1.0   train R@10 0.0392    <- worse than a fixed top-10 (0.0964)
#     TAU = 0.5   train R@10 0.3022
#     TAU = 0.2   train R@10 0.5036
#     TAU = 0.1   train R@10 0.5158    <- best
#     TAU = 0.05  train R@10 0.2700    <- over-sharpened, gradient saturates
#
# On the full data, 3 epochs: R@10 0.1218 at TAU 1.0 against 0.1916 at TAU 0.1.
TAU = 0.1

# EPOCHS was 3 because the untempered loss plateaued after ~500 steps. That is
# no longer true, so the run is length-limited by early stopping instead: train
# until test Recall@10 has not improved for PATIENCE epochs, then restore the
# best weights. Stopping on the metric rather than the loss matters here --
# under the old loss the two moved in opposite directions.
PATIENCE = 5

# logQ correction. In-batch negatives are the *targets of other rows*, so an
# item is drawn as a negative in proportion to how often it appears as a target
# -- frequent items are penalised far more often than rare ones, purely as an
# artifact of sampling. Subtracting log P(sampled) from each candidate's logit
# removes that bias (Yi et al., 2019). It matters here because error analysis
# showed recall is almost entirely a function of target frequency: targets seen
# >100 times score R@10 0.48, targets seen <=5 times score ~0.02.
#
# Applied only to the in-batch term. The in-node negatives are drawn uniformly
# within a category, so their sampling probability carries no per-item bias and
# needs no correction.
LOGQ_CORRECTION = True
torch.manual_seed(SEED)
print(f"device: {DEVICE}")

# --- Feature tensors, one row per item ------------------------------------
title_t = torch.tensor(arrays["title_emb"], device=DEVICE)
cat_t = torch.tensor(arrays["cat_ids"], device=DEVICE)
num_t = torch.tensor(arrays["numeric"], device=DEVICE)

# Standardise the numeric block on TRAIN items only, as in §§5-8.
train_item_idx = torch.tensor(
    np.unique(pairs_train[["query_idx", "target_idx"]].to_numpy()), device=DEVICE)
mu = num_t[train_item_idx].mean(0)
sd = num_t[train_item_idx].std(0).clamp_min(1e-6)
num_t = (num_t - mu) / sd


class ProductEncoder(nn.Module):
    """The paper's Product Encoder: attribute embeddings -> concat -> MLP."""

    def __init__(self):
        super().__init__()
        # padding_idx=0 keeps the reserved id inert -- it never receives gradient.
        self.embeddings = nn.ModuleList([
            nn.Embedding(len(vocabs[name]) + 1, CAT_DIM, padding_idx=0)
            for name in CAT_ORDER])
        self.title = nn.Linear(title_t.shape[1], 128)
        self.numeric = nn.Linear(num_t.shape[1], 16)
        self.mlp = nn.Sequential(
            nn.Linear(len(CAT_ORDER) * CAT_DIM + 128 + 16, HIDDEN),
            nn.ReLU(),
            nn.Linear(HIDDEN, OUT_DIM))

    def forward(self, idx):
        parts = [emb(cat_t[idx, j]) for j, emb in enumerate(self.embeddings)]
        parts += [self.title(title_t[idx]), self.numeric(num_t[idx])]
        return self.mlp(torch.cat(parts, dim=-1))


class ComplementaryTwoTower(nn.Module):
    """Same architecture both sides, no shared weights.

    Only the query tower sees the target-category embedding -- the one
    modification Complementary-TT makes to a plain two-tower.
    """

    def __init__(self):
        super().__init__()
        self.query_encoder = ProductEncoder()
        self.candidate_encoder = ProductEncoder()
        self.node = nn.Embedding(len(vocabs["target_node"]) + 1, NODE_DIM, padding_idx=0)
        self.query_out = nn.Linear(OUT_DIM + NODE_DIM, OUT_DIM)

    def query(self, idx, node_id):
        h = torch.cat([self.query_encoder(idx), self.node(node_id)], dim=-1)
        return F.normalize(self.query_out(h), dim=-1)

    def candidate(self, idx):
        return F.normalize(self.candidate_encoder(idx), dim=-1)


# --- Sampling probability of each item, as a target in training -----------
_target_freq = pairs_train.groupby("target_idx").size()
_count = np.zeros(len(cat_t), dtype="float64")
_count[_target_freq.index.to_numpy()] = _target_freq.to_numpy()
# An item never seen as a target is floored at 1 rather than dropped: it can
# still turn up as an in-batch negative, and log(0) is not a number.
LOGQ = torch.tensor(np.log(np.maximum(_count, 1.0) / max(_count.sum(), 1.0)),
                    dtype=torch.float32, device=DEVICE)
print(f"logQ spans {LOGQ.min():.2f} to {LOGQ.max():.2f} "
      f"({int((_count == 0).sum()):,} items never a target)")


# --- Hard negatives: items sharing the target's category ------------------
# Bucketed once by node id, so drawing one is two lookups and a uniform sample.
_order = np.argsort(node_of_item, kind="stable")
_starts = np.searchsorted(node_of_item[_order],
                          np.arange(node_of_item.max() + 2), side="left")
order_t = torch.tensor(_order, device=DEVICE)
starts_t = torch.tensor(_starts, device=DEVICE)


def hard_negatives(node_ids):
    lo, hi = starts_t[node_ids], starts_t[node_ids + 1]
    size = (hi - lo).clamp_min(1)
    offset = (torch.rand(len(node_ids), device=DEVICE) * size).long().clamp(max=size - 1)
    return order_t[(lo + offset).clamp(max=len(order_t) - 1)]


def bpr_loss(q, c_pos, c_hard=None, pos_idx=None):
    """BPR over in-batch negatives, plus one hard negative per row.

    `masked_fill`, not `masked_select`: a data-dependent output size is the
    pattern that returns the wrong element count on MPS.
    """
    scores = (q @ c_pos.T) / TAU
    if LOGQ_CORRECTION and pos_idx is not None:
        # column j is item pos_idx[j] acting as a negative for every other row
        scores = scores - LOGQ[pos_idx].unsqueeze(0)
    pos = scores.diagonal().unsqueeze(1)
    eye = torch.eye(len(q), dtype=torch.bool, device=q.device)
    easy = (-F.logsigmoid(pos - scores)).masked_fill(eye, 0.0).sum() / (len(q) * (len(q) - 1))
    if c_hard is None:
        return easy
    hard = -F.logsigmoid(pos.squeeze(1) - (q * c_hard).sum(-1) / TAU).mean()
    return (easy + hard) / 2


# --- Evaluation: rank the true target against the whole catalogue ---------
# Strict Recall@k credits only the target of the pair being scored. A test
# (query, category) key holds 1.73 held-out targets on average, and the rest sit
# in the candidate pool counted as wrong answers -- so strict recall understates
# top-10 quality by roughly a factor of two. `lenient=True` credits any of them.
# Both are reported, and the baselines below are measured the same way.
relevant = defaultdict(set)
for _q, _n, _t in pairs_test[["query_idx", "target_node_id", "target_idx"]].itertuples(index=False):
    relevant[(int(_q), int(_n))].add(int(_t))


@torch.no_grad()
def catalogue_vectors(model, chunk=8192):
    model.eval()
    out = torch.empty(len(cat_t), OUT_DIM, device=DEVICE)
    for i in range(0, len(cat_t), chunk):
        j = min(i + chunk, len(cat_t))
        out[i:j] = model.candidate(torch.arange(i, j, device=DEVICE))
    return out


@torch.no_grad()
def evaluate(model, pairs, k_values=(10, 20, 50, 100), n_eval=10_000, seed=0,
             lenient=False):
    cand = catalogue_vectors(model)
    take = np.random.default_rng(seed).choice(
        len(pairs), min(n_eval, len(pairs)), replace=False)
    sub = pairs.iloc[take]
    q_idx = torch.tensor(sub["query_idx"].to_numpy(), device=DEVICE)
    t_idx = torch.tensor(sub["target_idx"].to_numpy(), device=DEVICE)
    node = torch.tensor(sub["target_node_id"].to_numpy(), device=DEVICE)

    q_np, n_np = sub["query_idx"].to_numpy(), sub["target_node_id"].to_numpy()
    ranks = torch.empty(len(sub), dtype=torch.long, device=DEVICE)
    hit_lenient = np.zeros(len(sub), dtype=bool)
    for i in range(0, len(sub), 1024):
        j = min(i + 1024, len(sub))
        rows = torch.arange(j - i, device=DEVICE)
        scores = model.query(q_idx[i:j], node[i:j]) @ cand.T
        scores[rows, q_idx[i:j]] = -1e4                 # never recommend itself
        true = scores[rows, t_idx[i:j]].unsqueeze(1)
        ranks[i:j] = (scores > true).sum(1)             # 0-based rank
        if lenient:
            top = torch.topk(scores, 10, dim=1).indices.cpu().numpy()
            for r in range(j - i):
                hit_lenient[i + r] = bool(
                    set(top[r]) & relevant[(int(q_np[i + r]), int(n_np[i + r]))])
    model.train()

    out = {}
    for k in k_values:
        hit = (ranks < k).float()
        out[f"Recall@{k}"] = hit.mean().item()
        out[f"NDCG@{k}"] = (hit / torch.log2(ranks.float() + 2)).mean().item()
    out["MedianRank"] = float(ranks.median().item())
    if lenient:
        out["Recall@10_lenient"] = float(hit_lenient.mean())
    return out


# --- Baselines, scored exactly the way the model is ------------------------
# `10 / len(items)` was the wrong yardstick: the query tower is handed the true
# target category and 99.6% of its top-10 lands inside it, so the floor is a
# random draw WITHIN that category and the bar to clear is per-category
# popularity -- one groupby, no model. See ttn/experiments/ for what happened
# when the negative sampling was changed to try to beat it.
def baselines(pairs, n_eval=10_000, seed=0):
    take = np.random.default_rng(seed).choice(
        len(pairs), min(n_eval, len(pairs)), replace=False)
    sub = pairs.iloc[take]
    node = sub["target_node_id"].to_numpy()
    target = sub["target_idx"].to_numpy()
    query = sub["query_idx"].to_numpy()
    size = np.bincount(node_of_item, minlength=int(node_of_item.max()) + 1)

    counts = (pairs_train.groupby(["target_node_id", "target_idx"]).size()
              .rename("n").reset_index())
    top10 = {int(k): set(g.nlargest(10, "n")["target_idx"].astype(int))
             for k, g in counts.groupby("target_node_id")}
    return {
        "random over the whole catalogue": 10 / len(cat_t),
        "random within the asked-for category":
            float(np.mean(np.minimum(10 / size[node], 1))),
        "popularity within the category":
            float(np.mean([int(t) in top10.get(int(nd), set())
                           for nd, t in zip(node, target)])),
        "popularity within the category (lenient)":
            float(np.mean([bool(top10.get(int(nd), set()) & relevant[(int(q), int(nd))])
                           for q, nd in zip(query, node)])),
    }


# --- Train ----------------------------------------------------------------
model = ComplementaryTwoTower().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

q_all = torch.tensor(pairs_train["query_idx"].to_numpy(), device=DEVICE)
c_all = torch.tensor(pairs_train["target_idx"].to_numpy(), device=DEVICE)
n_all = torch.tensor(pairs_train["target_node_id"].to_numpy(), device=DEVICE)
print(f"training on {len(q_all):,} pairs over {len(cat_t):,} items\n")

# A shuffled pass per epoch, so every pair is seen exactly once each time.
# Sampling with replacement instead would leave ~16% of pairs unseen at this
# length and make two runs at the same step count disagree on their data.
steps_per_epoch = (len(q_all) + BATCH - 1) // BATCH
print(f"{EPOCHS} epochs x {steps_per_epoch:,} steps of {BATCH:,}")
m = evaluate(model, pairs_test, n_eval=5000)
print(f"epoch 0 (untrained)          R@10 {m['Recall@10']:.4f}  "
      f"R@100 {m['Recall@100']:.4f}  medRank {m['MedianRank']:,.0f}")

start = time.time()
step = 0
best = {"epoch": 0, "recall": -1.0, "state": None}
for epoch in range(1, EPOCHS + 1):
    order = torch.randperm(len(q_all), device=DEVICE)
    for i in range(0, len(q_all), BATCH):
        batch = order[i:i + BATCH]
        if len(batch) < 2:                 # in-batch negatives need a partner
            continue
        q = model.query(q_all[batch], n_all[batch])
        loss = bpr_loss(q, model.candidate(c_all[batch]),
                        model.candidate(hard_negatives(n_all[batch])),
                        pos_idx=c_all[batch])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        step += 1
    m = evaluate(model, pairs_test, n_eval=5000)
    mark = ""
    if m["Recall@10"] > best["recall"]:
        best = {"epoch": epoch, "recall": m["Recall@10"],
                "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}}
        mark = "  <- best"
    print(f"epoch {epoch:>2} ({step:,} steps)  loss {loss.item():.4f}  "
          f"R@10 {m['Recall@10']:.4f}  R@100 {m['Recall@100']:.4f}  "
          f"medRank {m['MedianRank']:,.0f}  [{time.time() - start:.0f}s]{mark}")
    if epoch - best["epoch"] >= PATIENCE:
        print(f"\nno improvement in {PATIENCE} epochs -- stopping at epoch {epoch}")
        break

# Report and save the BEST epoch, not the last one.
if best["state"] is not None:
    model.load_state_dict(best["state"])
    print(f"restored epoch {best['epoch']} (test R@10 {best['recall']:.4f})")

elapsed = time.time() - start
print(f"\n{EPOCHS} epochs / {step:,} steps in {elapsed:.0f}s "
      f"({len(q_all) * EPOCHS / elapsed:,.0f} pairs/s)\n")
metrics = {}
for split, pairs in (("train", pairs_train), ("test", pairs_test)):
    metrics[split] = evaluate(model, pairs, lenient=(split == "test"))
    print(f"{split:<6} " + "  ".join(
        f"{k} {v:,.0f}" if "Rank" in k else f"{k} {v:.4f}"
        for k, v in metrics[split].items()))
print()
for _name, _value in baselines(pairs_test).items():
    print(f"baseline  {_name:<42} {_value:.4f}")

# --- Save the weights, and everything needed to rebuild the model ---------
# The fitted artifacts (vocabs.json, the medians) already live on disk; without
# this the weights were the one half of the setup that vanished on restart, so
# no two sessions could compare models. The config travels with the state dict
# because ComplementaryTwoTower's shapes are derived from the vocabularies --
# reloading against a different §6 fit would silently mismatch.
CHECKPOINT = OUT_DIR / "ttn_complementary.pt"
torch.save({
    "state_dict": model.state_dict(),
    "config": {"cat_dim": CAT_DIM, "node_dim": NODE_DIM, "hidden": HIDDEN,
               "out_dim": OUT_DIM, "batch": BATCH, "epochs": EPOCHS, "lr": LR,
               "seed": SEED, "cat_order": CAT_ORDER,
               "logq_correction": LOGQ_CORRECTION,
               "tau": TAU, "patience": PATIENCE,
               "best_epoch": best["epoch"]},
    "vocab_sizes": {k: len(v) for k, v in vocabs.items()},
    "numeric_standardisation": {"mean": mu.cpu(), "std": sd.cpu()},
    "metrics": metrics,
    "n_items": int(len(cat_t)),
    "n_train_pairs": int(len(pairs_train)),
}, CHECKPOINT)
print(f"\nsaved -> {CHECKPOINT.relative_to(ROOT)} "
      f"({CHECKPOINT.stat().st_size / 1e6:,.1f} MB)")

# Reload into a fresh model and confirm it scores identically, so a restart
# genuinely resumes rather than appearing to.
checkpoint = torch.load(CHECKPOINT, weights_only=False)
reloaded = ComplementaryTwoTower().to(DEVICE)
reloaded.load_state_dict(checkpoint["state_dict"])
before = evaluate(model, pairs_test, n_eval=5000)["Recall@10"]
after = evaluate(reloaded, pairs_test, n_eval=5000)["Recall@10"]
assert abs(before - after) < 1e-9, f"reload changed the model: {before} vs {after}"
print(f"reloaded checkpoint reproduces R@10 exactly: {after:.4f}")

## 11. What does it actually recommend?

Take a query from the test set and read the top 10. This is not decoration —
it answers a question `Recall@10` structurally cannot.

**The thing to rule out.** The query tower is handed the target category, and
the model beats random-within-that-category by 4.6×. A model that had learned
nothing about complementarity, and instead simply ranked the *popular* items in
the requested category, would score much the same way — popular items are
popular because they are co-purchased often, so they turn up as targets often.
Aggregate metrics cannot separate "knows what completes this product" from
"knows what sells in this aisle".

So each query prints three things beside the recommendations:

- **the true target's rank**, so you can see the hit or miss directly;
- **`in_cat`** — how much of the top 10 falls in the requested category. Near
  100% every time means the node embedding is doing the work and the product
  features are along for the ride;
- **overlap with the popularity baseline** — the most frequent targets in that
  same category, counted from the training pairs. High overlap is the failure
  mode above; low overlap with comparable accuracy means the model is
  discriminating between items, which is the whole point.

Read the titles too. A mug query returning coasters, trays and a kettle is
complementary; returning four more mugs would mean §8's same-category filter
did not go far enough.


In [91]:
# --- Popularity baseline: most frequent targets, per category -------------
target_counts = (pairs_train.groupby(["target_node_id", "target_idx"])
                 .size().rename("n").reset_index())
popular_by_node = {node: g.nlargest(50, "n")["target_idx"].to_numpy()
                   for node, g in target_counts.groupby("target_node_id")}

catalogue = catalogue_vectors(model)          # encode every item once


@torch.no_grad()
def recommend(query_idx, node_id, k=10):
    q = model.query(torch.tensor([query_idx], device=DEVICE),
                    torch.tensor([node_id], device=DEVICE))
    scores = (q @ catalogue.T).squeeze(0)
    scores[query_idx] = -1e4                  # never recommend the query itself
    top = torch.topk(scores, k)
    return top.indices.cpu().numpy(), top.values.cpu().numpy()


def describe(idx):
    row = items.iloc[idx]
    price = f"${row['price']:,.2f}" if pd.notna(row["price"]) else "  --   "
    return (f"{str(row['title'])[:52]:<52} {price:>9}  "
            f"{str(row['cat_3'])[:22]:<22} {str(row['cat_4'])[:22]}")


def show(pair_row, k=10):
    q_idx = int(pair_row["query_idx"])
    t_idx = int(pair_row["target_idx"])
    node = int(pair_row["target_node_id"])
    top, scores = recommend(q_idx, node, k)

    node_name = [n for n, i in vocabs["target_node"].items() if i == node]
    print("QUERY    ", describe(q_idx))
    print("WANTED   ", describe(t_idx))
    print(f"asked for: {node_name[0] if node_name else '?'}")

    full = (model.query(torch.tensor([q_idx], device=DEVICE),
                        torch.tensor([node], device=DEVICE)) @ catalogue.T).squeeze(0)
    full[q_idx] = -1e4
    rank = int((full > full[t_idx]).sum())
    print(f"true target rank: {rank:,} of {len(catalogue):,}"
          f"{'   <-- HIT in top 10' if rank < 10 else ''}")

    in_cat = float(np.mean(node_of_item[top] == node))
    pop = popular_by_node.get(node, np.array([], dtype=int))[:k]
    overlap = len(set(top) & set(pop)) / k
    print(f"top-{k}: {in_cat:.0%} in the asked-for category | "
          f"{overlap:.0%} overlap with that category's most-popular items")
    for r, (i, s) in enumerate(zip(top, scores), 1):
        flag = "*" if i == t_idx else (" p" if i in set(pop) else "  ")
        print(f"  {r:>2}{flag} {s:+.3f}  {describe(int(i))}")
    print()


rng = np.random.default_rng(7)
for row in range(3):
    show(pairs_test.iloc[int(rng.integers(len(pairs_test)))])

# --- The same two diagnostics, aggregated over many queries ---------------
sample = pairs_test.iloc[rng.choice(len(pairs_test), 500, replace=False)]
in_cat, overlap = [], []
for _, r in sample.iterrows():
    node = int(r["target_node_id"])
    top, _ = recommend(int(r["query_idx"]), node, 10)
    in_cat.append(np.mean(node_of_item[top] == node))
    pop = set(popular_by_node.get(node, np.array([], dtype=int))[:10])
    overlap.append(len(set(top) & pop) / 10)
print(f"over 500 test queries:")
print(f"  top-10 in the asked-for category : {np.mean(in_cat):.1%}")
print(f"  top-10 overlap with popularity   : {np.mean(overlap):.1%}")
print("  (high overlap would mean the model is ranking by popularity within\n"
      "   the category rather than by what complements the query)")

QUERY     ironwork sateen woven blackout grommet top curtain p    $45.89  Window Treatments      Draperies & Curtains
WANTED    montevilla beme international 5 8 core ribbed knob t    $17.10  Window Treatment Hardw Window Rods
asked for: Home Dcor > Window Treatment Hardware > Window Rods
true target rank: 15 of 137,362
top-10: 100% in the asked-for category | 50% overlap with that category's most-popular items
   1 p +0.912  decopolitan urn telescoping double drapery rod set 7    $23.66  Window Treatment Hardw Window Rods
   2 p +0.894  decopolitan 1 inch urn single window treatment rod s    $34.56  Window Treatment Hardw Window Rods
   3 p +0.869  kenney valencia 5 8 rdquo standard decorative window    $22.08  Window Treatment Hardw Window Rods
   4 p +0.868  kenney 5 8 beckett decorative window curtain rod 48-    $13.96  Window Treatment Hardw Window Rods
   5 p +0.863  kenney chelsea 5 8 standard decorative window curtai    $21.67  Window Treatment Hardw Window Rods
   6   +0.842  

### Recommend for a given asin

The serving path. With a real product there is no "true target" to look up, so
the **category to retrieve from has to come from the mapping** — that is what
`complementary_categories.pkl` is for, and it is the input the query tower
expects alongside the product.

So this asks the mapping which categories the item's own category buys into,
takes the best-evidenced few, and runs the model once per category.

Categories are ranked by `edges` rather than `lift`. Both already cleared their
thresholds when the mapping was built, but lift alone promotes the thinnest
evidence: for a vegetable peeler it put *Fryers* (26 edges) and *Bread
Machines* (13 edges) ahead of *Fruit & Vegetable Tools* (362), because a pair
between two otherwise-quiet categories scores near the maximum on lift. `edges`
asks how much co-purchase traffic actually stands behind the pair. Self
pairs are excluded, since §8 dropped them from training.

Reading the output: each block is one *aisle* the mapping licensed, and the
ordering within it is the model's contribution. The merged list shares the 10 slots out **in proportion to each category's
co-purchase traffic**, and lets the model pick which items fill them.

That replaces a max over every category-conditioned score, which was unsound
twice over: those scores answer different questions and are not comparable, and
the model's entire top-10 span is about 0.003 — roughly one degree of angle — so
the ordering across categories was noise. It also let a 19-edge relation compete
head-on with a 217-edge one, which is how bed frames surfaced for a bookshelf.

Weighting the scores by evidence instead would have worked arithmetically but
not in spirit: the log-share gap between a strong and a weak category is ~2.4
against a 0.003 score span, so the category term outweighs the model by roughly
800× and the ranking would be evidence alone. Slots keep both contributions
real — evidence decides *how many* from each aisle, the model decides *which*.

Each category's picks are restricted to items actually in it, so a slot means
what it says.

Each line carries the asin and the product image url beneath it, for the query
and every recommendation. The urls come from `df_features` rather than
`items.npz` — they are for looking at, not for the model — and about half the
catalogue has no image, which prints as `(no image)`.


In [92]:
from complementary_cats_pairs import first_image_url

asin_to_idx = dict(zip(items["asin"], items["idx"]))

# Image urls are display-only, so they are looked up here rather than carried
# into items.npz. imageURL/imageURLHighRes are stringified lists of several
# shots of the same product; `first_image_url` takes the high-res one where it
# exists. Only ~49% of items have any image at all.
image_by_asin = pd.Series(
    first_image_url(df_features["imageURLHighRes"], df_features["imageURL"]).to_numpy(),
    index=df_features["asin"].to_numpy(),
)


def describe_with_image(idx):
    """Same line as `describe`, plus the product image url on the next line."""
    asin = items.loc[idx, "asin"]
    url = image_by_asin.get(asin)
    shown = url if isinstance(url, str) else "(no image)"
    return f"{describe(idx)}\n           {asin}  {shown}"


def licensed_targets(query_idx):
    """Category paths the mapping says this item's category buys into."""
    row = items.loc[query_idx]
    hit = comp_cat[
        (comp_cat["src_cat_2"].astype(str) == str(row["cat_2"]))
        & (comp_cat["src_cat_3"].astype(str) == str(row["cat_3"]))
        & (comp_cat["src_cat_4"].astype(str) == str(row["cat_4"]))].copy()
    hit["node"] = (hit["dst_cat_2"].astype(str) + NODE_SEP
                   + hit["dst_cat_3"].astype(str) + NODE_SEP
                   + hit["dst_cat_4"].astype(str))
    own = NODE_SEP.join(str(row[c]) for c in ("cat_2", "cat_3", "cat_4"))
    hit = hit[hit["node"] != own]                    # §8 dropped self pairs
    hit["node_id"] = hit["node"].map(vocabs["target_node"])
    # Ranked by `edges`, not `lift`. Lift alone puts the thinnest evidence
    # first -- a pair seen 13 times between two otherwise-quiet categories
    # scores near the maximum, which is the failure the package README warns
    # about. `edges` asks how much co-purchase traffic actually backs the pair;
    # both metrics already cleared their floors when the mapping was filtered.
    return hit.dropna(subset=["node_id"]).sort_values("edges", ascending=False)


def allocate_slots(shares, k):
    """Largest-remainder apportionment: k slots split by share, summing to k."""
    raw = np.asarray(shares, dtype=float) * k
    base = np.floor(raw).astype(int)
    for j in np.argsort(-(raw - base))[:k - base.sum()]:
        base[j] += 1
    return base


@torch.no_grad()
def recommend_in_category(query_idx, node_id, k):
    """Top-k restricted to items that actually sit in `node_id`."""
    q = model.query(torch.tensor([query_idx], device=DEVICE),
                    torch.tensor([node_id], device=DEVICE))
    scores = (q @ catalogue.T).squeeze(0)
    in_node = torch.tensor(node_of_item == node_id, device=DEVICE)
    scores = torch.where(in_node, scores, torch.full_like(scores, -1e4))
    scores[query_idx] = -1e4
    top = torch.topk(scores, min(k, max(int(in_node.sum()), 1)))
    return top.indices.cpu().numpy(), top.values.cpu().numpy()


@torch.no_grad()
def recommend_for_asin(asin, k=10, max_categories=3):
    if asin not in asin_to_idx:
        print(f"{asin!r} is not in the pair tables — it has no features, or no "
              f"co-purchase pair survived the §4 mapping filter.")
        return
    q_idx = int(asin_to_idx[asin])
    row = items.loc[q_idx]
    print("=" * 108)
    print(f"QUERY  {describe_with_image(q_idx)}")

    targets = licensed_targets(q_idx)
    if targets.empty:
        print("  the mapping licenses no complementary category for this item")
        return
    print(f"mapping licenses {len(targets)} target categories; "
          f"showing the {min(max_categories, len(targets))} best-evidenced\n")

    for _, t in targets.head(max_categories).iterrows():
        node_id = int(t["node_id"])
        top, scores = recommend(q_idx, node_id, k)
        print(f"--- {t['node']}   (lift {t['lift']:,.1f}, {int(t['edges']):,} edges) ---")
        for r, (i, s) in enumerate(zip(top, scores), 1):
            print(f"  {r:>2}  {s:+.3f}  {describe_with_image(int(i))}")
        print()

    # --- Merged, by proportional slots -----------------------------------
    # A max over every category-conditioned score would rank by numbers that
    # are not comparable across conditionings: the model's whole top-10 span is
    # ~0.003 (about 1 degree of angle), so the ordering there is noise, and a
    # 19-edge category competes on equal terms with a 217-edge one. Instead each
    # category gets a share of the k slots proportional to its co-purchase
    # traffic, and the model picks which items fill them. Scores are only ever
    # compared within a category, which is the only place they mean anything.
    shares = (targets["edges"] / targets["edges"].sum()).to_numpy()
    slots = allocate_slots(shares, k)

    print(f"--- merged: {k} slots shared out over "
          f"{int((slots > 0).sum())} of {len(targets)} categories by evidence ---")
    rank = 0
    for (_, t), n_slots in zip(targets.iterrows(), slots):
        if n_slots == 0:
            continue
        idx, scores = recommend_in_category(q_idx, int(t["node_id"]), int(n_slots))
        print(f"  [{t['dst_cat_4']}  {t['edges']:,} edges, "
              f"{t['edges'] / targets['edges'].sum():.1%} -> {n_slots} slot(s)]")
        for i, sc in zip(idx, scores):
            rank += 1
            print(f"  {rank:>2}  {sc:+.3f}  {describe_with_image(int(i))}")


# Pick an example that actually has complements to show.
EXAMPLE_ASIN = 'B000TDMUDS'
print(f"example asin: {EXAMPLE_ASIN}\n")
recommend_for_asin(EXAMPLE_ASIN)

# Change this to any asin in `items` and re-run:
#   recommend_for_asin("B00X5ETKU4")

example asin: B000TDMUDS

QUERY  winsome 92815 eugene accent table espresso              $67.00  Bedroom Furniture      Nightstands
           B000TDMUDS  https://images-na.ssl-images-amazon.com/images/I/41kw7xWV8iL.jpg
mapping licenses 15 target categories; showing the 3 best-evidenced

--- Furniture > Bedroom Furniture > Dressers   (lift 283.8, 185 edges) ---
   1  +0.837  sauder 409714 shoal creek 4-drawer chest l 34.72 x w    $28.40  Bedroom Furniture      Dressers
           B0042K14EA  https://images-na.ssl-images-amazon.com/images/I/51EgHrH76uL.jpg
   2  +0.824  south shore libra 4-drawer dresser pure black metal     $14.20  Bedroom Furniture      Dressers
           B00C0SDMAA  https://images-na.ssl-images-amazon.com/images/I/51DNIa4ZoXL.jpg
   3  +0.803  south shore libra 3-drawer dresser chocolate metal h    $28.40  Bedroom Furniture      Dressers
           B004WMYM4C  https://images-na.ssl-images-amazon.com/images/I/4160LvR6%2B2L.jpg
   4  +0.803  south shore libra 3-drawer

### The same thing, with pictures

`show_recommendations` renders the query and its recommendations as an image
grid instead of text. Same proportional-slot allocation as the cell above, so
the two agree — this is a different view of it, not a different result.

Images are fetched from Amazon's CDN when the cell renders, so it needs a
network connection, and about half of items have no image and show a grey
placeholder. Titles are HTML-escaped, since product titles are full of
quotes and angle brackets.


In [93]:
import html as _html
from IPython.display import HTML


def _card(idx, caption="", score=None):
    row = items.iloc[idx]
    url = image_by_asin.get(row["asin"])
    pic = (f'<img src="{_html.escape(str(url))}" style="width:120px;height:120px;'
           f'object-fit:contain;background:#fff;border-radius:4px">'
           if isinstance(url, str) else
           '<div style="width:120px;height:120px;background:#eee;border-radius:4px;'
           'display:flex;align-items:center;justify-content:center;color:#999;'
           'font-size:11px">no image</div>')
    price = f"${row['price']:,.2f}" if pd.notna(row["price"]) else "—"
    head = f'<div style="font-size:11px;color:#c60">{_html.escape(caption)}</div>' if caption else ""
    sc = f'<span style="color:#888"> {score:+.3f}</span>' if score is not None else ""
    return (f'<div style="width:132px;margin:6px;font-family:system-ui,sans-serif">'
            f'{head}{pic}'
            f'<div style="font-size:11px;line-height:1.3;margin-top:4px;height:46px;'
            f'overflow:hidden">{_html.escape(str(row["title"])[:70])}</div>'
            f'<div style="font-size:11px;color:#333"><b>{price}</b>{sc}</div>'
            f'<div style="font-size:10px;color:#888">{_html.escape(str(row["cat_4"])[:24])}</div>'
            f'</div>')


@torch.no_grad()
def show_recommendations(asin, k=10):
    """The proportional-slot recommendations for `asin`, as an image grid."""
    if asin not in asin_to_idx:
        return HTML(f"<p><code>{_html.escape(asin)}</code> is not in the pair tables.</p>")
    q_idx = int(asin_to_idx[asin])
    targets = licensed_targets(q_idx)
    if targets.empty:
        return HTML("<p>the mapping licenses no complementary category "
                    "for this item</p>")

    slots = allocate_slots((targets["edges"] / targets["edges"].sum()).to_numpy(), k)
    cards = []
    for (_, t), n in zip(targets.iterrows(), slots):
        if n == 0:
            continue
        idx, scores = recommend_in_category(q_idx, int(t["node_id"]), int(n))
        share = t["edges"] / targets["edges"].sum()
        for j, (i, sc) in enumerate(zip(idx, scores)):
            cards.append(_card(int(i),
                               f"{t['dst_cat_4']} · {share:.0%}" if j == 0 else "",
                               float(sc)))

    return HTML(
        '<div style="font-family:system-ui,sans-serif">'
        '<div style="font-size:13px;font-weight:600;margin-bottom:4px">QUERY</div>'
        f'<div style="display:flex">{_card(q_idx)}</div>'
        f'<div style="font-size:13px;font-weight:600;margin:10px 0 4px">'
        f'RECOMMENDED &mdash; {k} slots over {int((slots > 0).sum())} of '
        f'{len(targets)} licensed categories, shared by co-purchase evidence</div>'
        f'<div style="display:flex;flex-wrap:wrap">{"".join(cards)}</div></div>')


# --- The product to show. Any asin in `items`; defaults to the one the cell
# --- above picked, so the text and the pictures describe the same product.
SHOW_ASIN = 'B000GVB9W6'

display(show_recommendations(SHOW_ASIN))

In [94]:
print(f'Shape of the table: {tower_pairs_train.shape}')
print(f'Variables of the table: {tower_pairs_train.columns}')

Shape of the table: (2223995, 25)
Variables of the table: Index(['asin_query', 'query_cat_2', 'query_cat_3', 'query_cat_4',
       'query_title_cleaned', 'query_price', 'query_brand_clean',
       'query_color', 'query_features', 'query_material', 'query_product_type',
       'asin_target', 'target_cat_2', 'target_cat_3', 'target_cat_4',
       'target_title_cleaned', 'target_price', 'target_brand_clean',
       'target_color', 'target_features', 'target_material',
       'target_product_type', 'query_price_imputed', 'target_price_imputed',
       'target_node'],
      dtype='object')


## 12. Diversity audit — how many distinct items does a category get?

The complaint this answers: every item in a category coming back with the same
ten recommendations. `distinct` is counted against `slots` (queries x 10). One
shared list per category pins `distinct` at 10 however many queries are asked;
a model that actually reads the query pushes it toward `slots`.

The input category is the query item's own node, taken from `node_of_item` so
that it is named from the same vocabulary as the asked-for category.

In [95]:
# --- Diversity audit: how many DISTINCT items does a whole category get? ---
# The symptom this measures directly: every query item in a given category
# coming back with the same ten recommendations. A retrieval model conditioned
# on the query should not do that, so this quantifies it rather than leaving it
# to the eye.
#
# A cell below is one (input category, asked-for category) pair. The input
# category is the QUERY item's own node -- read from `node_of_item`, the same
# vocabulary the asked-for category uses, so both ends are named alike.
#
# Read `distinct` against `slots`: n queries x 10 recommendations. If every
# query got its own list, distinct would approach slots. If the category gets
# one shared list, distinct lands on 10 no matter how many queries there are.
K = 10
MAX_QUERIES = 150      # queries sampled per cell
MIN_QUERIES = 20       # cells smaller than this cannot say anything
MAX_JACCARD = 60       # pairwise Jaccard is O(n^2); cap the lists compared
TOP_CELLS = 25         # busiest cells, by distinct query items

node_name = {i: n for n, i in vocabs["target_node"].items()}
# The last path element alone is ambiguous: "Missing" is the folded catch-all
# from S7 and appears under many different parents, so two rows labelled
# "Missing" need not be the same node. Show the last two levels.
def short(nid):
    parts = str(node_name.get(int(nid), "?")).split(" > ")
    return " > ".join(p[:16] for p in parts[-2:])


@torch.no_grad()
def top_k_batch(q_idx, node_ids, k=K, chunk=512):
    """Top-k for many (query, asked-for category) rows at once."""
    model.eval()
    out = []
    for i in range(0, len(q_idx), chunk):
        qi = torch.tensor(np.asarray(q_idx[i:i + chunk]), device=DEVICE)
        nd = torch.tensor(np.asarray(node_ids[i:i + chunk]), device=DEVICE)
        scores = model.query(qi, nd) @ catalogue.T
        scores[torch.arange(len(qi), device=DEVICE), qi] = -1e4
        out.append(torch.topk(scores, k, dim=1).indices.cpu().numpy())
    model.train()
    return np.concatenate(out)


def mean_jaccard(sets, seed=0):
    """Mean pairwise overlap of the top-10 lists. 1.0 = every query identical."""
    if len(sets) < 2:
        return float("nan")
    if len(sets) > MAX_JACCARD:
        pick = np.random.default_rng(seed).choice(len(sets), MAX_JACCARD, replace=False)
        sets = [sets[i] for i in pick]
    return float(np.mean([len(a & b) / len(a | b)
                          for i, a in enumerate(sets) for b in sets[i + 1:]]))


probe = pairs_test[["query_idx", "target_node_id"]].copy()
probe["query_node"] = node_of_item[probe["query_idx"].to_numpy()]
cell_size = probe.groupby(["query_node", "target_node_id"])["query_idx"].nunique()
cells = cell_size[cell_size >= MIN_QUERIES].sort_values(ascending=False).index[:TOP_CELLS]

rows = []
for qnode, tnode in cells:
    g = probe[(probe["query_node"] == qnode) & (probe["target_node_id"] == tnode)]
    qs = pd.unique(g["query_idx"].to_numpy())[:MAX_QUERIES]
    tops = top_k_batch(qs, np.full(len(qs), int(tnode)))
    sets = [set(t) for t in tops]
    rows.append({
        "input category": short(qnode),
        "asked for": short(tnode),
        "queries": len(qs),
        "slots": len(qs) * K,
        "distinct": int(len(np.unique(tops))),
        "distinct %": len(np.unique(tops)) / (len(qs) * K),
        "distinct top-1": int(len(np.unique(tops[:, 0]))),
        "jaccard@10": mean_jaccard(sets),
        "items available": int((node_of_item == int(tnode)).sum()),
    })

audit = pd.DataFrame(rows).sort_values("distinct %")
print("Per (input category -> asked-for category), least diverse first:\n")
print(audit.to_string(index=False, float_format=lambda x: f"{x:,.3f}"))

# --- Rolled up to the input category the question was asked about ----------
roll = (audit.groupby("input category")
        .agg(cells=("asked for", "size"), queries=("queries", "sum"),
             slots=("slots", "sum"), distinct=("distinct", "sum"),
             jaccard=("jaccard@10", "mean"))
        .assign(**{"distinct %": lambda d: d["distinct"] / d["slots"]})
        .sort_values("distinct %"))
print("\n\nRolled up by input category:\n")
print(roll.to_string(float_format=lambda x: f"{x:,.3f}"))

identical = (audit["distinct"] <= K).sum()
print(f"\n\nOverall across the {len(audit)} busiest cells")
print(f"  mean jaccard@10 between two queries in the same cell : {audit['jaccard@10'].mean():.3f}")
print(f"  distinct items returned, as a share of slots         : "
      f"{audit['distinct'].sum() / audit['slots'].sum():.3f}")
print(f"  cells returning literally one shared list (<= {K} distinct): {identical} of {len(audit)}")
print(f"  median distinct top-1 across cells                   : {audit['distinct top-1'].median():,.0f}")
print("\njaccard 1.0 and distinct == 10 mean the query item changed nothing.")
print("jaccard near 0 and distinct near slots mean every query got its own list.")

Per (input category -> asked-for category), least diverse first:

                     input category                           asked for  queries  slots  distinct  distinct %  distinct top-1  jaccard@10  items available
Decorative Pillo > Throw Pillow Cov Window Treatment > Draperies & Curt      150   1500        36       0.024               6       0.650             2399
Window Treatment > Draperies & Curt      Window Treatment > Window Rods      150   1500        38       0.025               8       0.608              391
       Area Rugs, Runne > Area Rugs           Living Room Furn > Tables      150   1500        40       0.027              11       0.526             1084
Window Treatment > Draperies & Curt Sheets & Pillowc > Sheet & Pillowca      150   1500        43       0.029               9       0.490             1659
Bathroom Accesso > Shower Curtains,                 Bath Rugs > Missing      150   1500        45       0.030              13       0.579              675
Wind

In [96]:
tower_pairs_train[
                  # (tower_pairs_train['query_cat_2'] == 'Furniture') &
                  (tower_pairs_train['query_cat_3'] == 'Living Room Furniture') &
                  (tower_pairs_train['query_cat_4'] == 'Tables')]['asin_query'].iloc[131]

# tower_pairs_train.iloc[917221]['asin_query']

'B000GVB9W6'

## 13. Results log — what was measured, and what it means

Written after a round of experiments in September 2026. Every number below was
measured on this data; raw logs are in `ttn/experiments/`. The point of this
section is that a future reader should not have to re-run the dead ends.

### Resolved: the loss had no temperature

Everything below this section was measured **before** that was found, and the
model then scored 0.1179. The defect was one line. Both towers are L2-normalised
so scores live in [-1, 1], and `score(top1) - score(top10)` was **0.0067** —
squarely in the linear region of `logsigmoid`, where a badly-ordered pair pulls
no harder than a well-ordered one. The loss fell while the ranking stayed
arbitrary.

Isolated on 5,000 pairs inside ONE category, 150 epochs (so the node embedding
is constant and cannot help). The ceiling for a model emitting one fixed top-10
is 0.0964:

| TAU | train R@10 |
| --- | --- |
| 1.0 (no temperature) | 0.0392 — **worse than a constant** |
| 0.5 | 0.3022 |
| 0.2 | 0.5036 |
| **0.1** | **0.5158** |
| 0.05 | 0.2700 — over-sharpened |

An inverted U, 13x at the optimum. On the full data with `TAU = 0.1`, 30 epochs
and early stopping (best was epoch 9):

| | before | after | popularity baseline |
| --- | --- | --- | --- |
| Recall@10 | 0.1179 | **0.2174** | 0.2136 |
| Recall@10 lenient | 0.2506 | **0.4115** | 0.4034 |
| Recall@100 | 0.4634 | **0.5723** | 0.5837 |
| median rank | 120 | **69** | — |

**+84% on R@10, and it now clears per-category popularity on both strict and
lenient.** Train 0.2161 against test 0.2174, so it is signal rather than
memorisation. Recall@100 is still short of popularity.

**What this did not fix: diversity.** Distinct share stayed at 0.048 against
0.044, and pairwise Jaccard did not fall. Temperature fixed the *ranking*
defect, not the *collapse* defect — different queries in a category still get
near-identical lists, now better ordered. The 13.5x node dominance is untouched
and remains the open cause. §12 is the measurement to watch.

**Why this was missed for so long:** every temperature setting in the ablation
below was bundled with mined negatives, which were independently halving
performance. TAU 0.2 scored 0.0810 there and was read as "temperature does not
rescue mining" — when in fact it was the right knob attached to the wrong one.
Change one thing at a time.

### Where the model stands

| | R@10 | R@100 | median rank | R@10 lenient |
| --- | --- | --- | --- | --- |
| random over the whole catalogue | 0.0001 | — | — | — |
| random within the asked-for category | 0.0253 | — | — | — |
| raw title-embedding cosine, **untrained** | 0.0757 | 0.3065 | — | — |
| **this model** | **0.1179** | 0.4634 | 120 | **0.2506** |
| **popularity within the category** (one `groupby`) | **0.2136** | — | — | **0.4034** |

Two facts to hold together. The model clears the sensible floors — it beats a
random in-category draw 4.7x, and it beats untrained cosine similarity — but it
**loses to per-category popularity**, which needs no model at all. Re-running the
control gives 0.115–0.124, so treat ±0.005 as noise.

Train R@10 is 0.1183 against test 0.1179: a **generalisation gap of 0.0004**.
Nothing here is overfitting, so regularisation and more data are irrelevant.

### The symptom, measured

Section 12 quantifies the complaint that started this: every item in a category
returning the same recommendations.

* Across the 25 busiest (input category → asked-for category) cells, the model
  returns **4.4% as many distinct items as it has slots** — 3,750 queries x 10
  produced roughly 1,700 distinct items.
* Mean Jaccard between two queries in the same cell: **0.46**.
* Median **8 distinct top-1 items** across 150 different query products.
* Worst cell: 150 Cooking Utensils asking for Measuring Tools & Scales returned
  **17 distinct items** and **4 distinct top-1s**, out of 1,272 available.

### What was tried, and what happened

| experiment | result | verdict |
| --- | --- | --- |
| negatives mined from the model's own in-node top-K | R@10 0.1240 → **0.0636** | **much worse** |
| + temperature 0.2 / 0.05 on the cosine scores | 0.0810 / 0.0659 | still worse |
| + LayerNorm on the query blocks | 0.0660 | still worse |
| uniform in-node negatives: 1 → 4 → 16 → 64 | 0.1191 / 0.1159 / 0.1143 / 0.1205 | **no effect** |
| the same, measured on diversity | distinct share 0.042 / 0.044 / 0.043 / 0.044 | **no effect** |
| remove the target-category embedding | R@10 **0.0037**, median rank 4,058 | **catastrophic** |

**Mined negatives fail for a specific reason.** Mined negatives sit at the
**0.836** popularity percentile within their category; the true targets sit at
**0.832**; a uniform in-node draw sits at 0.500. Mining does not find hard-but-
wrong items, it finds *the items that are usually right* and trains the model to
reject them. Masking the 2.2M known `(query, target)` pairs does not rescue it —
only 1.07% of mined negatives are provably a complement of that query.

**Removing the node fails because the query cannot imply the category.** Without
it only **7.2%** of the top-10 lands in the asked-for category, against 99.2%
with it. Diversity does rise (Jaccard 0.44 → 0.09) but it is the diversity of
near-random retrieval. A query has a median of 3 target categories (mean 5.4),
so one unconditioned vector cannot serve them all.

**The features are not the problem.** Within a category, title embeddings average
**0.390** pairwise cosine (0.187 for two random items anywhere), brands number
142–533 per category at 4.4–6.7 bits of entropy, and price CV runs 0.5–2.05. A
grey table and a walnut table are not near-identical inputs. Untrained cosine on
those same features already scores 0.0757.

### The diagnosis that survives

The query tower collapses to roughly one direction per category:

* in `query_out`, the node branch reaches **13.5x** the norm of the item branch,
  leaving the query item to tilt the result about 4 degrees;
* two queries in the same node sit at cosine **0.99**;
* replacing the query item with a **uniformly random** one costs only
  0.1138 → 0.1073, so the query is worth ~6% of the score;
* the title block — the most discriminative input — enters the query MLP with
  the *smallest* norm of the four blocks (2.79, against 10.85 for attributes).

Loss changes cannot fix this, which is what the table above shows empirically.

### What `also_buy` actually is

Measured on the raw metadata field, not the derived pairs
(`notebooks/also_buy_popularity.ipynb`, `ttn/experiments/`):

* **36.8%** of edges are same-brand, against a 0.229% chance rate — a **161x
  lift**. much of `also_buy` is other products from the same line.
* **54.8%** are same `cat_4`. `also_buy` is a *co-purchase* field; it mixes
  complements with substitutes and does not distinguish them.
* It is **not** simply popularity: rebuilding each list from its category mix plus
  popularity recovers only **2.9%** of it, and two source items pointing at the
  same category overlap by Jaccard **0.005**. The raw field is highly
  item-specific.
* That specificity is concentrated in the same-category half — which §19 removes.
  After filtering, same-brand falls from 36.8% to **9.2%**.

**§19 filters very unevenly.** Because it drops same-`cat_4` pairs, it keeps 86%
of Knife Sets edges and **4%** of Incense & Incense Holders edges. The training
set is silently re-weighted toward functional-tool categories and nearly empty of
collectibles and consumables. Same-category rate also falls with price (0.613 in
the cheapest decile to 0.430 in the dearest, r = −0.123), but the category effect
(0.136 to 0.961) is far larger than the price effect.

### Corrections to earlier claims in this notebook

* §10 previously proposed mined negatives as the next step. They were tried and
  reverted; see the table above.
* A count-based test suggested that conditioning on the query adds nothing over
  the target category (0.2147 → 0.2039). That test is **sparsity-limited**: with
  raw `also_buy` at Jaccard 0.005 there are almost no repeated query→target
  observations to count, so it cannot distinguish "no signal" from "signal too
  sparse to count". Do not read it as proof of absence.
* The `random baseline Recall@10 = 10 / len(items)` line that used to close §10
  flattered the model ~1,600x. The query tower is handed the true category and
  99.6% of its top-10 lands inside it, so the floor is a random draw *within* the
  category, and the bar is per-category popularity.
* Strict Recall@k understates the model roughly 2x. A test `(query, category)`
  key holds 1.73 held-out targets on average and the ones not being scored are
  counted as wrong answers; `evaluate(..., lenient=True)` credits them.

### Open, in the order worth doing

1. **Can the model overfit a small subset?** Train on ~20k pairs for 50 epochs.
   The train/test parity above shows no memorisation, but 2.2M pairs is too many
   to memorise at this size, so it cannot separate *capacity* from *architecture*.
   Shrinking the data removes that ambiguity: if recall climbs toward 1.0 the
   architecture is fine and the label is the problem; if it plateaus near "the
   best single list per category", the node bottleneck is proven.
2. **Report the two subtasks separately.** Choosing the category (99.2%) and
   ranking inside it (worse than a `groupby`) are blended into one R@10 today,
   which hides both. Report lift over popularity, and the §12 diversity number,
   alongside recall.
3. **Decide §19 deliberately.** Either accept the uneven filtering and document
   it, or replace the blanket same-`cat_4` rule with a sharper substitute test —
   same `cat_4` **and** same brand, given the 161x brand lift. Also settle the
   `!= MISSING_LABEL` exemption, which currently leaks same-category pairs.
4. **Retarget on review co-occurrence.** The one available label that is not
   Amazon's own recommender output and is not dominated by same-brand variants.
   6.9M reviews over 777k users are in `Home_and_Kitchen_filtered.csv`. The
   conditional-signal test in `ttn/experiments/` is reusable as-is.

**Not worth more effort:** further negative-sampling changes (three independent
experiments eliminated them), and adding a popularity feature — it would raise
R@10 while making the diversity symptom worse, and the standard alternative is a
logQ correction in the sampling rather than popularity as an input.

## 14. Recall by category

One aggregate R@10 averages over 432 categories that behave very differently.
This breaks it out by the query's category, alongside the things that might
explain the spread: surviving training pairs, candidate-pool size, and price.

§19 is the first suspect. It drops same-`cat_4` pairs unevenly — 86% of Knife
Sets edges survive against 4% of Incense — so a category can score badly simply
because most of its data was filtered away.

In [97]:
# --- Recall by category ----------------------------------------------------
# One aggregate R@10 hides a lot. This breaks it out by the QUERY's category and
# puts next to it the things that plausibly explain the spread: how many training
# pairs survived §19, how many candidates the asked-for categories hold, and what
# the query items cost.
#
# Note what §19 does to this. It drops same-cat_4 pairs, and it does so very
# unevenly -- 86% of Knife Sets edges survive against 4% of Incense ones -- so a
# category can score poorly simply because almost none of its data is left.
MIN_TEST_PAIRS = 300

node_name = {i: n for n, i in vocabs["target_node"].items()}
price_of = np.expm1(arrays["numeric"][:, 0])

_probe = pairs_test[["query_idx", "target_idx", "target_node_id"]].copy()
_probe["query_node"] = node_of_item[_probe["query_idx"].to_numpy()]
_train_n = pd.Series(node_of_item[pairs_train["query_idx"].to_numpy()]).value_counts()
_node_sz = pd.Series(node_of_item).value_counts()

cand_v = catalogue_vectors(model)
model.eval()
rows = []
for qnode, g in _probe.groupby("query_node"):
    if len(g) < MIN_TEST_PAIRS:
        continue
    qi = torch.tensor(g["query_idx"].to_numpy(), device=DEVICE)
    ti = torch.tensor(g["target_idx"].to_numpy(), device=DEVICE)
    nd = torch.tensor(g["target_node_id"].to_numpy(), device=DEVICE)
    ranks = torch.empty(len(g), dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        for i in range(0, len(g), 1024):
            j = min(i + 1024, len(g))
            r = torch.arange(j - i, device=DEVICE)
            sc = model.query(qi[i:j], nd[i:j]) @ cand_v.T
            sc[r, qi[i:j]] = -1e4
            ranks[i:j] = (sc > sc[r, ti[i:j]].unsqueeze(1)).sum(1)
    rows.append({
        "query category": " > ".join(str(node_name.get(int(qnode), "?")).split(" > ")[-2:])[:34],
        "test pairs": len(g),
        "train pairs": int(_train_n.get(int(qnode), 0)),
        "R@10": (ranks < 10).float().mean().item(),
        "R@100": (ranks < 100).float().mean().item(),
        "med rank": float(ranks.median()),
        "median $": float(np.median(price_of[g["query_idx"].to_numpy()])),
        "cands": int(np.median([_node_sz.get(int(n), 0) for n in g["target_node_id"]])),
    })
model.train()

by_cat = pd.DataFrame(rows).sort_values("R@10", ascending=False)
print(f"Recall by query category ({len(by_cat)} categories with >= {MIN_TEST_PAIRS} test pairs)\n")
print("BEST:")
print(by_cat.head(10).to_string(index=False, float_format=lambda x: f"{x:,.3f}"))
print("\nWORST:")
print(by_cat.tail(10).to_string(index=False, float_format=lambda x: f"{x:,.3f}"))
print(f"\nspread: R@10 from {by_cat['R@10'].min():.3f} to {by_cat['R@10'].max():.3f} "
      f"(overall {(by_cat['R@10'] * by_cat['test pairs']).sum() / by_cat['test pairs'].sum():.3f})")
print("\nwhat explains it -- correlation of R@10 with:")
for col in ("train pairs", "median $", "cands", "test pairs"):
    x = np.log1p(by_cat[col].to_numpy()) if col != "median $" else np.log(by_cat[col].clip(0.01))
    print(f"   log({col:<12}) {np.corrcoef(x, by_cat['R@10'])[0, 1]:+.3f}")
print("\nA strong negative correlation with `cands` means the model does well where")
print("there is little to choose between; a positive one with `train pairs` means")
print("categories gutted by §19 are the ones that score badly.")

Recall by query category (141 categories with >= 300 test pairs)

BEST:
                    query category  test pairs  train pairs  R@10  R@100  med rank  median $  cands
Cutlery & Knife Accessories > Chef         640         6045 0.444  0.783    16.000    44.990    230
Cutlery & Knife Accessories > Pari         389         3487 0.398  0.746    21.000    11.990    383
Cutlery & Knife Accessories > Asia         370         3382 0.397  0.770    17.000    44.990    293
Cutlery & Knife Accessories > Knif         578         6626 0.389  0.734    20.000    14.950    293
Kitchen & Table Linens > Dish Clot         564         7308 0.385  0.688    24.000    13.440    497
               Cookware > Griddles         343         3806 0.353  0.679    26.000    14.880    622
Cutlery & Knife Accessories > Miss         319         3003 0.345  0.730    30.000    33.480    293
      Bakeware > Bread & Loaf Pans         388         3975 0.340  0.732    29.000    15.990    514
   Cookware > Cookware Acces

In [101]:
by_cat[by_cat['query category'].str.contains('Furniture', na=False)]

,query category,test pairs,train pairs,R@10,R@100,med rank,median $,cands
26,"Bedroom Furniture > Beds, Frames &",2235,18003,0.31,0.72,27.00,28.39,576
35,Living Room Furniture > TV & Media,1157,7822,0.26,0.64,48.00,28.39,576
27,Bedroom Furniture > Dressers,330,2696,0.25,0.64,45.00,28.40,576
36,Living Room Furniture > Tables,2050,14176,0.25,0.63,49.00,21.29,379
28,Bedroom Furniture > Mattresses & B,2059,12919,0.25,0.65,40.00,49.99,987
33,Home Office Furniture > Home Offic,643,5157,0.24,0.60,60.00,14.20,451
29,Bedroom Furniture > Nightstands,371,2608,0.22,0.63,55.00,21.29,661
31,Home Office Furniture > Bookcases,530,4093,0.21,0.54,72.00,19.25,600
30,Game & Recreation Room Furniture >,433,4245,0.15,0.46,123.00,14.20,1084
32,Home Office Furniture > Home Offic,378,2778,0.15,0.48,108.00,14.20,600


## 15. Re-ranking the retrieved candidates

The tower scores by a dot product of two independently encoded vectors, so it
cannot compute *"does this share the query's material"* — neither tower sees the
other side. That is what a second pass adds, and it is why a walnut table can be
returned white bar chairs however well the tower is trained.

Scale is the thing to get right. The whole gap from rank 1 to rank 10 is about
0.076 of cosine, so a bonus of 0.15 would leap a candidate over the entire top
ten — replacing the model with a rule rather than refining it. The sweep is
centred an order of magnitude below that.

Recall@100 is the ceiling: re-ranking reorders what was retrieved, it never
retrieves anything new.

In [102]:
# --- Re-ranking the top-100 on attribute agreement -------------------------
# The tower scores by a dot product of two INDEPENDENTLY encoded vectors, so it
# structurally cannot compute "does the target share the query's material" --
# neither tower sees the other side. A second pass over the retrieved candidates
# can, which is the one thing this stage adds that no amount of tower training
# would.
#
# Scale matters more than sign here. Measured on the trained model, the whole
# gap from rank 1 to rank 10 is ~0.076 of cosine, so a bonus of 0.15 would leap a
# candidate over the entire top ten and the rule would replace the model rather
# than refine it. The sweep below is centred an order of magnitude lower.
#
# The ceiling is Recall@100: re-ranking reorders the retrieved set, it never
# retrieves anything new.
N_EVAL, DEPTH = 5000, 100

_rng = np.random.default_rng(0)
_sub = pairs_test.iloc[_rng.choice(len(pairs_test), N_EVAL, replace=False)]
_q = _sub["query_idx"].to_numpy(); _t = _sub["target_idx"].to_numpy()
_n = _sub["target_node_id"].to_numpy()

# --- retrieve once; the sweep is then pure numpy --------------------------
_cand_v = catalogue_vectors(model)
model.eval()
_cands, _ann = [], []
with torch.no_grad():
    for i in range(0, N_EVAL, 512):
        j = min(i + 512, N_EVAL)
        qi = torch.tensor(_q[i:j], device=DEVICE)
        sc = model.query(qi, torch.tensor(_n[i:j], device=DEVICE)) @ _cand_v.T
        sc[torch.arange(j - i, device=DEVICE), qi] = -1e4
        top = torch.topk(sc, DEPTH, dim=1)
        _cands.append(top.indices.cpu().numpy()); _ann.append(top.values.cpu().numpy())
model.train()
cands = np.concatenate(_cands); ann = np.concatenate(_ann)

# --- the cross-features the tower cannot see ------------------------------
_col = lambda f: arrays["cat_ids"][:, CAT_ORDER.index(f)]
_price = np.expm1(arrays["numeric"][:, 0])
_imputed = arrays["numeric"][:, 2] > 0.5          # §7 median-imputed: no real price

def _match(field):
    c = _col(field); miss = vocabs[field].get("Missing", -1)
    ok = (c[_q] != miss) & (c[_q] != 0)
    return ((c[cands] == c[_q][:, None]) & ok[:, None]).astype("float32")

same_material = _match("material")
same_brand = _match("brand")
# Price gap, hinged: free within e^1 (~2.7x), penalised beyond. Imputed prices
# are excluded rather than scored, so the rule cannot reward our own imputation.
with np.errstate(divide="ignore", invalid="ignore"):
    _gap = np.abs(np.log(np.maximum(_price[cands], 1e-6) / np.maximum(_price[_q][:, None], 1e-6)))
price_pen = np.where(_imputed[cands] | _imputed[_q][:, None], 0.0,
                     np.maximum(0.0, _gap - 1.0)).astype("float32")
print(f"candidates sharing the query's material: {same_material.mean():.1%}   "
      f"brand: {same_brand.mean():.1%}   price usable: {(price_pen >= 0).mean():.0%}")

def recall_at(w_mat, w_brand, w_price, ks=(10, 20)):
    s = ann + w_mat * same_material + w_brand * same_brand - w_price * price_pen
    order = np.argsort(-s, axis=1)
    hit_rank = np.argmax(cands[np.arange(len(cands))[:, None], order] == _t[:, None], axis=1)
    found = (cands == _t[:, None]).any(1)
    return {f"R@{k}": float(((hit_rank < k) & found).mean()) for k in ks}

base = recall_at(0, 0, 0)
print(f"\nbaseline (tower order): R@10 {base['R@10']:.4f}  R@20 {base['R@20']:.4f}  "
      f"ceiling R@{DEPTH} {float((cands == _t[:, None]).any(1).mean()):.4f}\n")

# Negative weights included deliberately: if the tower already over-selects
# same-material candidates, the useful correction is to penalise the match,
# not reward it. A one-sided grid would never find that.
grid = [-0.08, -0.04, -0.02, -0.01, 0.0, 0.005, 0.01, 0.02, 0.04, 0.08]
print("one coefficient at a time (others zero):")
print(f"{'w':>7} {'material R@10':>15} {'brand R@10':>12} {'price R@10':>12}")
for w in grid:
    print(f"{w:>7.3f} {recall_at(w,0,0)['R@10']:>15.4f} {recall_at(0,w,0)['R@10']:>12.4f} "
          f"{recall_at(0,0,w)['R@10']:>12.4f}")

best, rows = None, []
for wm in grid:
    for wb in grid:
        for wp in grid:
            r = recall_at(wm, wb, wp)
            rows.append((wm, wb, wp, r["R@10"], r["R@20"]))
            if best is None or r["R@10"] > best[3]:
                best = (wm, wb, wp, r["R@10"], r["R@20"])
sweep = pd.DataFrame(rows, columns=["w_material", "w_brand", "w_price", "R@10", "R@20"])
print(f"\nbest of {len(rows)} combinations: material {best[0]}, brand {best[1]}, "
      f"price {best[2]}\n  R@10 {best[3]:.4f} (baseline {base['R@10']:.4f}, "
      f"{best[3] - base['R@10']:+.4f})   R@20 {best[4]:.4f} (baseline {base['R@20']:.4f}, "
      f"{best[4] - base['R@20']:+.4f})")
print("\ntop 5 by R@10:")
print(sweep.nlargest(5, "R@10").to_string(index=False, float_format=lambda x: f"{x:,.4f}"))
print("\nA gain here is real but bounded by Recall@100 -- and hand-set weights")
print("optimise the label, which barely rewards coordination (material 1.27x,")
print("brand 1.41x over chance). If you want coordination for product reasons")
print("beyond what the sweep picks, that is a deliberate trade against R@10.")

candidates sharing the query's material: 11.8%   brand: 10.6%   price usable: 100%

baseline (tower order): R@10 0.2264  R@20 0.3130  ceiling R@100 0.5756

one coefficient at a time (others zero):
      w   material R@10   brand R@10   price R@10
 -0.080          0.2168       0.2104       0.2126
 -0.040          0.2240       0.2190       0.2232
 -0.020          0.2240       0.2232       0.2260
 -0.010          0.2254       0.2244       0.2268
  0.000          0.2264       0.2264       0.2264
  0.005          0.2260       0.2266       0.2266
  0.010          0.2266       0.2268       0.2272
  0.020          0.2258       0.2282       0.2272
  0.040          0.2240       0.2298       0.2252
  0.080          0.2210       0.2252       0.2216

best of 1000 combinations: material 0.0, brand 0.04, price 0.01
  R@10 0.2310 (baseline 0.2264, +0.0046)   R@20 0.3152 (baseline 0.3130, +0.0022)

top 5 by R@10:
 w_material  w_brand  w_price   R@10   R@20
     0.0000   0.0400   0.0100 0.2310 0.3152
  

## 16. What did the model learn about colour?

Every categorical field is an `nn.Embedding`, so the vector the model learned for
`dark walnut` can be read straight off the weight matrix. This section pulls the
colour table out, shows it as a dataframe you can index by value, and projects it
to two dimensions to see which colours the model placed near each other.

Two things to keep in mind while reading it.

**These are not semantic embeddings.** Nothing told the model that navy is a kind
of blue. The geometry is whatever minimised the co-purchase loss, so `black` next
to `dark walnut` is a statement about what gets bought together, not about
colour.

**Check `n_items` before trusting a neighbour.** A value carried by thirty items
gets thirty items' worth of gradient; its vector is largely noise. The same tail
problem §18 measures at the item level shows up inside the vocabulary.

In [ ]:
# --- Reading the learned attribute embeddings ------------------------------
# Every categorical field is an `nn.Embedding`, so the vector the model learned
# for "dark walnut" can be read straight off the weight matrix. The two towers
# share no weights, so each has its OWN table for the same vocabulary -- switch
# EMB_TOWER to compare them.
#
# These are NOT semantic embeddings. Nothing told the model that navy is a kind
# of blue; the geometry is whatever minimised the co-purchase loss. Values that
# appear on few items get few gradient updates, so their vectors are mostly
# noise -- read the `n_items` column before trusting a neighbour.
import matplotlib.pyplot as plt

EMB_TOWER = "candidate"          # or "query"


def embedding_table(field, tower=EMB_TOWER):
    """One row per vocabulary value: the value, how many items carry it, its vector."""
    enc = model.candidate_encoder if tower == "candidate" else model.query_encoder
    W = enc.embeddings[CAT_ORDER.index(field)].weight.detach().cpu().numpy()
    inv = {i: v for v, i in vocabs[field].items()}
    counts = np.bincount(arrays["cat_ids"][:, CAT_ORDER.index(field)], minlength=W.shape[0])
    df = pd.DataFrame([{"value": inv.get(i, "(reserved id 0)"), "id": i,
                        "n_items": int(counts[i]), "embedding": W[i]}
                       for i in range(W.shape[0])])
    return df.sort_values("n_items", ascending=False).reset_index(drop=True)


def _pca_2d(X):
    """Two principal components, and the share of variance they carry."""
    Xc = X - X.mean(0)
    _, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    return Xc @ Vt[:2].T, float((S[:2] ** 2).sum() / (S ** 2).sum())


def plot_embeddings(field, top=40, method="pca", tower=EMB_TOWER, figsize=(11, 8)):
    """Scatter the `top` most common values of `field` in two dimensions."""
    df = embedding_table(field, tower)
    d = df[df.id > 0].head(top).reset_index(drop=True)   # id 0 is the reserved slot
    X = np.stack(d.embedding.to_numpy())
    var = None
    if method == "umap":
        try:
            import umap
            XY = umap.UMAP(n_neighbors=min(15, len(d) - 1), random_state=SEED).fit_transform(X)
        except ImportError:
            print("umap-learn not installed — falling back to PCA")
            method = "pca"
    if method == "pca":
        XY, var = _pca_2d(X)
    fig, ax = plt.subplots(figsize=figsize)
    size = 40 + 260 * (np.log1p(d.n_items) / np.log1p(d.n_items).max())
    ax.scatter(XY[:, 0], XY[:, 1], s=size, alpha=0.35, edgecolor="none")
    for i, row in d.iterrows():
        ax.annotate(f"{row['value']}", (XY[i, 0], XY[i, 1]), fontsize=8,
                    xytext=(4, 3), textcoords="offset points")
    sub = f"  (PCA, {var:.0%} of variance)" if var is not None else "  (UMAP)"
    ax.set_title(f"{field} embeddings, {tower} tower — {len(d)} most common values{sub}\n"
                 f"marker size = how many items carry the value", fontsize=11)
    ax.set_xlabel("component 1"); ax.set_ylabel("component 2")
    ax.grid(alpha=0.15)
    plt.tight_layout(); plt.show()
    return d.assign(x=XY[:, 0], y=XY[:, 1])


def nearest_values(field, value, k=6, tower=EMB_TOWER, min_items=25):
    """The k closest values by cosine — what the model thinks is 'like' this one.

    `min_items` matters more than it looks. A value carried by one or two items
    has had almost no gradient and its vector is close to its initialisation, so
    it lands near everything by accident. Without a floor the neighbour lists
    fill up with them: `metal` reads as nearest to "polyester alternative"
    (1 item) and "japanese vg-10" (2 items).
    """
    df = embedding_table(field, tower)
    df = df[(df.n_items >= min_items) | (df.value == value)].set_index("value")
    if value not in df.index:
        raise KeyError(f"{value!r} is not in the {field} vocabulary")
    X = np.stack(df.embedding.to_numpy())
    X = X / np.linalg.norm(X, axis=1, keepdims=True).clip(1e-9)
    s = X @ X[df.index.get_loc(value)]
    order = np.argsort(-s)[1:k + 1]
    return pd.DataFrame({"value": df.index[order], "cosine": s[order],
                         "n_items": df.n_items.to_numpy()[order]})


# --- The table: one row per colour, with its vector ------------------------
color_embeddings = embedding_table("color")
print(f"colour vocabulary: {len(color_embeddings) - 1} values + the reserved id 0, "
      f"{color_embeddings.embedding.iloc[0].shape[0]}-d each\n")
_view = color_embeddings.head(25).copy()
_view["embedding (first 6 of 32)"] = _view.embedding.apply(
    lambda v: " ".join(f"{x:+.2f}" for x in v[:6]) + " …")
print(_view[["value", "id", "n_items", "embedding (first 6 of 32)"]]
      .to_string(index=False))
print("\nThe full 32-d vector for any colour:  "
      "color_embeddings.set_index('value').loc['black', 'embedding']")

### The colour space in two dimensions

PCA by default: deterministic, and the variance share says how much structure
survived the projection. `method="umap"` works if `umap-learn` is installed — it
preserves local neighbourhoods better, but its axes mean nothing and the layout
shifts with its parameters.

In [ ]:
# --- Two dimensions, so the geometry can be looked at ----------------------
# PCA is the default because it is deterministic and the variance share tells
# you how much of the structure survived the projection. Pass method="umap" if
# umap-learn is installed; it preserves local neighbourhoods better but the
# axes mean nothing and the layout changes with its parameters.
color_2d = plot_embeddings("color", top=40, method="pca")

# Neighbours are easier to judge than a scatter. Check a few by hand:
for _c in ("black", "white", "red", "chrome"):
    if _c in set(color_embeddings.value):
        print(f"\nnearest to {_c!r}:")
        print(nearest_values("color", _c, k=5).to_string(index=False))

## 17. The same for material

Identical treatment for `material`. The helpers from §16 (`embedding_table`,
`plot_embeddings`, `nearest_values`) work on any categorical field, so
`plot_embeddings("brand")` or `plot_embeddings("product_type")` will also run.

In [ ]:
# --- The same for material -------------------------------------------------
material_embeddings = embedding_table("material")
print(f"material vocabulary: {len(material_embeddings) - 1} values + the reserved id 0\n")
_view = material_embeddings.head(25).copy()
_view["embedding (first 6 of 32)"] = _view.embedding.apply(
    lambda v: " ".join(f"{x:+.2f}" for x in v[:6]) + " …")
print(_view[["value", "id", "n_items", "embedding (first 6 of 32)"]].to_string(index=False))

material_2d = plot_embeddings("material", top=40, method="pca")

for _m in ("wood", "metal", "glass", "cotton", "stainless steel"):
    if _m in set(material_embeddings.value):
        print(f"\nnearest to {_m!r}:")
        print(nearest_values("material", _m, k=5).to_string(index=False))

## 18. Error analysis — where the model fails, and why

Measured September 2026 on the tempered model (`TAU = 0.1`), 20,000 sampled test
pairs. Logs in `ttn/experiments/`. §13 records what was tried and reverted; this
section records what the remaining failures actually are.

### Recall is almost entirely a function of how often the target was seen

| target's frequency as a target in TRAIN | pairs | R@10 |
| --- | --- | --- |
| never | 668 | 0.0030 |
| 1–2 | 962 | 0.0146 |
| 3–5 | 1,214 | 0.0198 |
| 6–10 | 1,610 | 0.0329 |
| 11–25 | 2,985 | 0.0586 |
| 26–100 | 5,722 | 0.1414 |
| **>100** | **6,839** | **0.4816** |

A **160x spread**. About a third of test pairs have well-observed targets and
score 0.48; the other two thirds have rare targets and score 0.01–0.14. The
headline 0.22 is an average over two very different populations.

The hit/miss profile says the same thing — target frequency separates them 8x,
while attribute agreement differs by about five points:

| | hit | miss |
| --- | --- | --- |
| target's train frequency (median) | 239 | 31 |
| target never seen as a target | 0.0% | 4.3% |
| candidates in the target category | 463 | 1,067 |
| same brand as query | 18.5% | 13.0% |
| same material as query | 25.2% | 20.4% |

**Cold queries are fine; cold targets are not.** A query item unseen in training
scores 0.2067 against 0.2190 for a seen one — no real difference. The asymmetry
is entirely on the candidate side, where rare items' embeddings receive too few
gradient updates to land anywhere useful.

### A third of the hits come from pairs already seen

| split | pairs | model | popularity |
| --- | --- | --- | --- |
| exact (query, target) also in train | 2,817 (14.1%) | **0.5087** | 0.4622 |
| **genuinely unseen** | 17,183 (85.9%) | **0.1710** | **0.1742** |
| all | 20,000 | 0.2185 | 0.2147 |

This is not leakage — the split is temporal, and the same two items genuinely
being bought together in both periods is real signal. But it means the honest
generalisation figure is **0.171**, and on genuinely new pairs the model is
**level with, or marginally behind, per-category popularity**.

### The query item does matter — it just doesn't buy more than popularity

| | R@10 |
| --- | --- |
| real query item | 0.2185 |
| **uniformly random** query item | 0.1709 |
| the query is worth | **+0.0476 (22% of the score)** |

Before the temperature fix this was +0.0065, or 6%. So `TAU` did not only raise
recall, it made the model actually read its query — a 7x increase in the query's
contribution. Overlap with the popularity top-10 is 53.8%, so the model is
finding *different* items of *similar* quality, not better ones.

Blending the two helps a little and is bounded: the union of the model's top-10
and popularity's top-10 contains the target **26.9%** of the time, so no
combination of these two lists can beat that. A swept linear blend
(`score + w·log1p(popularity)`) peaks at **0.2243** with w = 0.02, up 0.006.

### The category whitelist admits about a fifth of behaviour

Cell 9 keeps a behavioural co-purchase pair only if its category combination
appears in `complementary_categories.pkl`, which `categories.py` derives from
Amazon's `also_buy` lists with a lift >= 2, edges >= 5 rule.

| | |
| --- | --- |
| behavioural pairs the whitelist licenses | **20.6% train / 21.3% test** |
| per query item, share of its target categories licensed | **18.5%** |
| query items with **none** of their categories licensed | **46.1%** |
| end to end | 10,595,885 raw → 2,916,635 after the whitelist → 2,223,995 after §19 |

The filter runs before the split is used, so the discarded 79% is invisible to
training **and** to every metric here. Category pairs that pass the *same*
lift >= 2, edges >= 5 test on behavioural data but are absent from the whitelist
include Living Room Tables → Area Rugs (2,889 edges, lift 2.5), Bed Pillows →
Sheet Sets (2,290, 2.2) and Candles → Incense (3,122, 2.6). 7,672 such pairs
exist; only 23.5% of behaviourally-qualified pairs are licensed.

**Replacing the whitelist with a behavioural one is not the fix** — the same rule
computed on behaviour licenses only **7.3%** of edges against the current 18.9%,
because most behavioural co-occurrence has lift below 2. The union of the two is
the direction worth testing, screened for substitutes as §19 screens them.

### What follows

1. **Rare-target representation** is the largest lever. Two thirds of test pairs
   have targets seen 25 times or fewer, and they score under 0.06. Re-ranking and
   popularity blending cannot help there — those items never reach the top-100 to
   be reordered. A logQ correction on the in-batch term is the first move and is
   implemented in §10.
2. **Report recall split by target frequency**, not as a single number. A change
   that lifts the cold two thirds while costing the head is an improvement, and
   an aggregate figure will read it as a regression.
3. **Widen the category whitelist**, and judge it on coverage rather than on
   R@10 — the task gets harder, so recall will fall even if the system improves.
4. **Diversity remains unfixed.** §12 still measures distinct-share at 0.048 with
   a median of 8 distinct top-1 items across 150 query products. Temperature
   fixed ranking, not collapse; the 13.5x node dominance in `query_out` is
   untouched, and LayerNorm or a smaller `NODE_DIM` has never been tested in
   isolation.
5. **Fix the early-stopping leak.** §10 selects its best epoch on test
   Recall@10, which is model selection on the evaluation set. Carve a validation
   slice out of the pre-cutoff period and report test once.